# 04 — Минимальный диагностический запуск OpenML-кандидатов

Цель: выполнить первый безопасный технический запуск трех OpenML-кандидатов после утверждения протоколов признаков, разбиений, моделей и метрик.

Граница интерпретации: эта тетрадь не выбирает победителя, не меняет статусы кандидатов, не подтверждает финальные утверждения ML-CRA и не выполняет подбор гиперпараметров.

Проверяемые кандидаты:

1. `openml_electricity_151`;
2. `openml_miniboone_41150`;
3. `openml_click_prediction_small_41434`.

In [45]:
from __future__ import annotations

from pathlib import Path
import importlib
import importlib.util
import platform
import sys
import time
import warnings
from typing import Any

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 120)

RANDOM_SEED = 20260507
TEST_SIZE = 0.20

warnings.filterwarnings("default")

In [46]:
def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / "data_registry" / "dataset_candidate_registry.csv").exists():
            return path
    raise FileNotFoundError("Не найден data_registry/dataset_candidate_registry.csv")


PROJECT_ROOT = find_project_root()
DATA_REGISTRY_DIR = PROJECT_ROOT / "data_registry"
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

REGISTRY_PATH = DATA_REGISTRY_DIR / "dataset_candidate_registry.csv"
FEATURE_PLAN_PATH = DATA_REGISTRY_DIR / "openml_feature_preprocessing_plan.csv"
MODEL_PLAN_PATH = DATA_REGISTRY_DIR / "openml_model_protocol_plan.csv"
SPLIT_PLAN_PATH = DATA_REGISTRY_DIR / "openml_split_protocol_plan.csv"
METRIC_PLAN_PATH = DATA_REGISTRY_DIR / "openml_metric_plan.csv"
NOTEBOOK_PLAN_PATH = DATA_REGISTRY_DIR / "openml_first_diagnostic_notebook_plan.csv"

SCORES_PATH = DATA_REGISTRY_DIR / "openml_first_diagnostic_scores.csv"
WARNINGS_PATH = DATA_REGISTRY_DIR / "openml_first_diagnostic_warnings.csv"
ENVIRONMENT_PATH = DATA_REGISTRY_DIR / "openml_first_diagnostic_environment.csv"

required_paths = [
    REGISTRY_PATH,
    FEATURE_PLAN_PATH,
    MODEL_PLAN_PATH,
    SPLIT_PLAN_PATH,
    METRIC_PLAN_PATH,
    NOTEBOOK_PLAN_PATH,
]

missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError("Не найдены обязательные файлы протоколов: " + "; ".join(map(str, missing_paths)))

print("PROJECT_ROOT =", PROJECT_ROOT)
for path in required_paths:
    print("OK:", path.relative_to(PROJECT_ROOT))

PROJECT_ROOT = C:\Users\Vanargo\Desktop\ML-CRA
OK: data_registry\dataset_candidate_registry.csv
OK: data_registry\openml_feature_preprocessing_plan.csv
OK: data_registry\openml_model_protocol_plan.csv
OK: data_registry\openml_split_protocol_plan.csv
OK: data_registry\openml_metric_plan.csv
OK: data_registry\openml_first_diagnostic_notebook_plan.csv


In [47]:
# --- Временная диагностическая ячейка  --- #

print("Python:", sys.executable)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_DIR:", SRC_DIR)
print("SRC_DIR in sys.path:", str(SRC_DIR) in sys.path)

Python: c:\Users\Vanargo\Desktop\ML-CRA\ml-cra-venv\Scripts\python.exe
PROJECT_ROOT: C:\Users\Vanargo\Desktop\ML-CRA
SRC_DIR: C:\Users\Vanargo\Desktop\ML-CRA\src
SRC_DIR in sys.path: True


In [3]:
required_packages = ["openml", "sklearn", "numpy", "pandas"]
missing_packages = [name for name in required_packages if importlib.util.find_spec(name) is None]

if missing_packages:
    print("Не найдены обязательные пакеты:", missing_packages)
    print("Установи их в активном окружении, например: pip install openml scikit-learn pandas numpy")
    raise SystemExit("Остановлено: отсутствуют обязательные пакеты.")

import openml
import sklearn
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    f1_score,
    log_loss,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

print("openml =", getattr(openml, "__version__", "unknown"))
print("scikit-learn =", sklearn.__version__)
print("pandas =", pd.__version__)
print("numpy =", np.__version__)

openml = 0.15.1
scikit-learn = 1.8.0
pandas = 3.0.2
numpy = 2.4.4


In [4]:
def read_project_csv(path: Path) -> pd.DataFrame:
    try:
        return pd.read_csv(
            path,
            dtype=str,
            keep_default_na=False,
            encoding="utf-8-sig",
            sep=None,
            engine="python",
        )
    except pd.errors.ParserError as error:
        raise pd.errors.ParserError(
            f"Не удалось прочитать CSV-файл {path.relative_to(PROJECT_ROOT)}. "
            "Проверь разделитель, кавычки и количество полей в строках. "
            f"Исходная ошибка pandas: {error}"
        )


registry = read_project_csv(REGISTRY_PATH)
feature_plan = read_project_csv(FEATURE_PLAN_PATH)
model_plan = read_project_csv(MODEL_PLAN_PATH)
split_plan = read_project_csv(SPLIT_PLAN_PATH)
metric_plan = read_project_csv(METRIC_PLAN_PATH)
notebook_plan = read_project_csv(NOTEBOOK_PLAN_PATH)

candidate_ids = [
    "openml_electricity_151",
    "openml_miniboone_41150",
    "openml_click_prediction_small_41434",
]

required_registry_columns = [
    "candidate_id",
    "source_name",
    "source_dataset_id",
    "dataset_name",
    "target_name",
    "status",
]

required_feature_plan_columns = [
    "candidate_id",
    "feature_names",
    "first_contour_policy",
]

for column in required_registry_columns:
    if column not in registry.columns:
        raise KeyError(f"В {REGISTRY_PATH.relative_to(PROJECT_ROOT)} отсутствует обязательный столбец: {column}")

for column in required_feature_plan_columns:
    if column not in feature_plan.columns:
        raise KeyError(f"В {FEATURE_PLAN_PATH.relative_to(PROJECT_ROOT)} отсутствует обязательный столбец: {column}")

candidates = registry[registry["candidate_id"].isin(candidate_ids)].copy()
if len(candidates) != len(candidate_ids):
    found = set(candidates["candidate_id"])
    missing = [candidate_id for candidate_id in candidate_ids if candidate_id not in found]
    raise ValueError(f"В реестре не найдены кандидаты: {missing}")

print("Кандидаты первого диагностического запуска:")
display(candidates[required_registry_columns])

print("План признаков:")
display(feature_plan[feature_plan["candidate_id"].isin(candidate_ids + ["all_openml_candidates"])])

print("Протокол тетради прочитан:")
display(notebook_plan.head(10))

Кандидаты первого диагностического запуска:


,candidate_id,source_name,source_dataset_id,dataset_name,target_name,status
0,openml_electricity_151,OpenML,151,electricity,class,temporal_sensitivity_candidate
1,openml_miniboone_41150,OpenML,41150,MiniBooNE,signal,primary_tabular_candidate
2,openml_click_prediction_small_41434,OpenML,41434,Click_prediction_small,click,leakage_risk_diagnostic_hold


План признаков:


,candidate_id,feature_group_id,feature_names,feature_role_ru,first_contour_policy,preprocessing_policy_ru,allowed_models,blocked_models,fit_scope,main_risk_ru,archive_basis,external_basis,notes_ru
0,openml_electricity_151,electricity_regular_numeric_predictors,nswprice; nswdemand; vicprice; vicdemand; tran...,обычные числовые признаки для первого контура,use,для LogisticRegression — стандартизация внутри...,logistic_regression; hist_gradient_boosting; r...,,fit_only_on_train_inside_pipeline,если стандартизация будет выполнена до разбиен...,openml_feature_preview.csv lines 5-9; openml_d...,scikit-learn `Pipeline` (конвейер обработки) /...,целевые значения не используются в преобразова...
1,openml_electricity_151,electricity_temporal_context_fields,date; day; period,поля временного порядка и отдельной проверки ч...,split_order_only_primary; model_feature_later_...,в первом основном контуре использовать для вос...,not_applicable_for_primary_model_features,all_models_as_primary_predictors_until_review,not_fitted; used_for_split_order_review,сырые временные поля могут смешать проверку мо...,openml_data_preview_summary.csv line 2; openml...,scikit-learn common pitfalls (типичные ошибки)...,это консервативное решение; позже можно добави...
2,openml_miniboone_41150,miniboone_numeric_predictors,ParticleID_0 ... ParticleID_49,числовые признаки,use,для LogisticRegression — стандартизация внутри...,logistic_regression; hist_gradient_boosting; r...,,fit_only_on_train_inside_pipeline,высокая вычислительная нагрузка на 130064 стро...,openml_data_preview_summary.csv line 3; openml...,scikit-learn `Pipeline` (конвейер обработки) /...,признаки времени и идентификаторы не обнаружен...
3,openml_click_prediction_small_41434,click_safe_numeric_predictors,impression; depth; position,безопасные неидентификаторные числовые признаки,use_under_drop_all_identifiers_policy,для LogisticRegression — стандартизация внутри...,dummy_prior; logistic_regression; hist_gradien...,catboost_classifier; lightgbm_classifier; targ...,fit_only_on_train_inside_pipeline,качество может быть низким после исключения ид...,"openml_feature_preview.csv lines 60,64,65; cli...",scikit-learn `Pipeline` (конвейер обработки) /...,это основной режим первой версии для Click_pre...
4,openml_click_prediction_small_41434,click_identifier_features_drop_primary,url_hash; ad_id; advertiser_id; query_id; keyw...,идентификаторные признаки-кандидаты,drop_in_primary_safe_baseline,полностью исключить из основной первой постано...,none_in_primary_safe_baseline,all_models_if_identifiers_used_without_policy_...,not_fitted_in_primary_safe_baseline,модель может выигрывать за счет запоминания ид...,"openml_feature_preview.csv lines 61-63,66-70; ...",scikit-learn common pitfalls (типичные ошибки)...,диагностическое использование идентификаторов ...
5,all_openml_candidates,missing_values_policy,all_features,контроль пропусков,assert_zero_missing; stop_if_mismatch,не вводить автоматическое заполнение пропусков...,all_first_contour_models_after_assertion,all_models_if_missing_values_appear_without_pr...,not_applicable_before_mismatch,автоматическое заполнение пропусков без проток...,openml_data_preview_summary.csv lines 2-4 show...,scikit-learn common pitfalls (типичные ошибки)...,это правило защищает воспроизводимость; оно не...
6,all_openml_candidates,categorical_encoding_policy,low_cardinality_non_identifier_categorical_fea...,категориальные признаки без признаков-идентифи...,use_only_if_low_cardinality_and_non_identifier,использовать прямое кодирование категорий толь...,logistic_regression; random_forest_limited; hi...,target_encoding_in_first_contour,fit_only_on_train_inside_pipeline,кодирование категорий на полном наборе до разб...,click_prediction_identifier_policy.md lines 50...,scikit-learn `ColumnTransformer` (преобразоват...,для electricity day допустим только после отде...


Протокол тетради прочитан:


,step_id,target_object,cell_group_ru,planned_action_ru,must_run_models,allowed,blocked,archive_basis,external_basis,expected_future_output,notes_ru
0,nb00_target_notebook,notebooks/04_dataset_smoke_experiments.ipynb,выбор целевой тетради,"использовать существующую пустую тетрадь, пред...",no,yes,создание новой тетради 05 без необходимости,REPORT_07_dataset_candidate_registry_protocol....,not_applicable,not_applicable,само изменение тетради должно идти отдельным ш...
1,nb01_header_and_safety_notice,notebook_markdown_cell,заголовок и границы эксперимента,"зафиксировать, что это диагностический запуск,...",no,yes,интерпретация результатов как финальных доказа...,REPORT_21_openml_first_diagnostic_contour_plan...,not_applicable,human_readable_notebook_context,предупреждение должно быть в первой markdown-я...
2,nb02_imports_project_root,notebook_code_cell,импорты и корень проекта,"подготовить импорты, настройки pandas и функци...",no,yes,жесткие абсолютные пути локального компьютера,existing notebooks 01 and 02 use find_project_...,not_applicable,resolved_project_paths,точный код будет дан в следующем отчете
3,nb03_load_protocol_files,notebook_code_cell,загрузка протоколов проекта,"прочитать реестр кандидатов, планы предобработ...",no,yes,обход CSV-протоколов вручную в коде,REPORT_22_openml_diagnostic_plan_artifacts.md ...,not_applicable,loaded_protocol_tables,"тетрадь должна останавливаться, если обязатель..."
4,nb04_load_openml_candidates,notebook_code_cell,загрузка OpenML-кандидатов,"загрузить openml_electricity_151, openml_minib...",no,yes,загрузка новых кандидатов вне реестра,REPORT_21_openml_first_diagnostic_contour_plan...,OpenML Python client documentation if implemen...,loaded_X_y_per_candidate,"если OpenML-пакет не установлен, тетрадь должн..."
5,nb05_feature_policy_assertions,notebook_code_cell,контроль признаков и пропусков,собрать X строго по openml_feature_preprocessi...,no,yes,автоматическое заполнение пропусков без нового...,REPORT_25 lines 47-56; openml_feature_preproce...,scikit-learn common pitfalls documentation on ...,safe_feature_matrices,date/day/period не входят в X модели electrici...
6,nb06_split_primary,notebook_code_cell,основное разбиение данных,создать одно стратифицированное случайное разб...,no,yes,временное разбиение electricity без аудита пор...,"openml_split_protocol_plan.csv lines 2, 6, 10;...",scikit-learn StratifiedShuffleSplit documentation,train_test_indices,повторные разбиения и временной holdout остают...
7,nb07_model_recipes,notebook_code_cell,модельные рецепты,"подготовить DummyClassifier, LogisticRegressio...",no,yes,CatBoost; LightGBM; обязательный RandomForestC...,REPORT_24 lines 15-27; openml_model_protocol_p...,scikit-learn Pipeline documentation; scikit-le...,configured_estimators,масштабирование допустимо только внутри Pipeline
8,nb08_fit_score,notebook_code_cell,минимальное обучение и оценка,обучить модели на train-части и посчитать метр...,yes_later,yes_after_notebook_change,обучение до внесения утвержденного изменения т...,REPORT_21 lines 84-95; openml_metric_plan.csv ...,scikit-learn cross_validate documentation if c...,score_rows,"первый запуск — техническая связность, а не фи..."
9,nb09_write_outputs,notebook_code_cell,сохранение черновых диагностических выходов,"создать CSV с результатами, предупреждениями и...",no,yes,изменение dataset_candidate_registry.csv и кар...,REPORT_25 lines 109-119,not_applicable,data_registry/openml_first_diagnostic_scores.c...,выходы должны иметь статус draft / diagnostic


In [5]:
def split_semicolon_list(value: str) -> list[str]:
    return [item.strip() for item in str(value).split(";") if item.strip()]


def expand_feature_names(raw_feature_spec: str, available_columns: list[str]) -> list[str]:
    spec = str(raw_feature_spec).strip()
    if "..." not in spec:
        return split_semicolon_list(spec)

    if spec == "ParticleID_0 ... ParticleID_49":
        expanded = [f"ParticleID_{i}" for i in range(50)]
        missing = [col for col in expanded if col not in available_columns]
        if missing:
            raise KeyError(f"Не найдены ожидаемые признаки MiniBooNE: {missing}")
        return expanded

    raise ValueError(f"Неизвестная сокращенная спецификация признаков: {spec}")


def selected_feature_names(candidate_id: str, X: pd.DataFrame) -> list[str]:
    rows = feature_plan[
        (feature_plan["candidate_id"].eq(candidate_id))
        & (feature_plan["first_contour_policy"].isin(["use", "use_under_drop_all_identifiers_policy"]))
    ].copy()
    if rows.empty:
        raise ValueError(f"Не найдены разрешенные признаки первого контура для {candidate_id}")

    selected: list[str] = []
    for _, row in rows.iterrows():
        selected.extend(expand_feature_names(row["feature_names"], list(X.columns)))

    selected = list(dict.fromkeys(selected))
    missing = [col for col in selected if col not in X.columns]
    if missing:
        raise KeyError(f"В данных {candidate_id} отсутствуют признаки из протокола: {missing}")

    if candidate_id == "openml_click_prediction_small_41434":
        blocked_identifiers = {
            "url_hash", "ad_id", "advertiser_id", "query_id", "keyword_id", "title_id", "description_id", "user_id"
        }
        leakage_columns = sorted(blocked_identifiers.intersection(selected))
        if leakage_columns:
            raise ValueError(f"В признаки Click_prediction_small ошибочно попали идентификаторы: {leakage_columns}")

    if candidate_id == "openml_electricity_151":
        blocked_time_features = {"date", "day", "period"}
        leakage_columns = sorted(blocked_time_features.intersection(selected))
        if leakage_columns:
            raise ValueError(f"В признаки electricity ошибочно попали временные поля: {leakage_columns}")

    return selected


def assert_no_missing(candidate_id: str, X: pd.DataFrame, y: pd.Series) -> None:
    feature_missing = int(X.isna().sum().sum())
    target_missing = int(y.isna().sum())
    if feature_missing or target_missing:
        raise ValueError(
            f"{candidate_id}: обнаружены пропуски. "
            f"feature_missing={feature_missing}, target_missing={target_missing}. "
            "Первый контур должен быть остановлен до обновления протокола пропусков."
        )


def coerce_numeric_features(candidate_id: str, X: pd.DataFrame) -> pd.DataFrame:
    converted = X.copy()
    non_numeric_before = [col for col in converted.columns if not pd.api.types.is_numeric_dtype(converted[col])]
    for col in non_numeric_before:
        converted[col] = pd.to_numeric(converted[col], errors="raise")
    non_numeric_after = [col for col in converted.columns if not pd.api.types.is_numeric_dtype(converted[col])]
    if non_numeric_after:
        raise TypeError(f"{candidate_id}: после преобразования остались нечисловые признаки: {non_numeric_after}")
    return converted


def encode_binary_target(candidate_id: str, y: pd.Series) -> tuple[np.ndarray, dict[str, Any]]:
    y_series = pd.Series(y).copy()
    if y_series.isna().any():
        raise ValueError(f"{candidate_id}: целевая переменная содержит пропуски")

    encoder = LabelEncoder()
    y_encoded = encoder.fit_transform(y_series.astype(str))
    classes = list(encoder.classes_)
    if len(classes) != 2:
        raise ValueError(f"{candidate_id}: ожидалась бинарная классификация, получены классы: {classes}")

    metadata = {
        "target_class_0": classes[0],
        "target_class_1": classes[1],
        "positive_class_assumption": classes[1],
    }
    return y_encoded, metadata

In [6]:
loaded: dict[str, dict[str, Any]] = {}
warning_rows: list[dict[str, Any]] = []

for _, row in candidates.iterrows():
    candidate_id = row["candidate_id"]
    did = int(row["source_dataset_id"])
    expected_target = row["target_name"]

    print("\n" + "=" * 110)
    print(f"Загрузка: {candidate_id} | OpenML ID: {did}")

    dataset = openml.datasets.get_dataset(did)
    target_name = getattr(dataset, "default_target_attribute", None) or expected_target
    if expected_target and target_name != expected_target:
        warning_rows.append({
            "severity": "warning",
            "candidate_id": candidate_id,
            "object": "target_name",
            "warning_ru": f"Целевая переменная OpenML ({target_name}) отличается от реестра ({expected_target}). Используется значение OpenML.",
        })

    X_raw, y_raw, categorical_indicator, attribute_names = dataset.get_data(
        target=target_name,
        dataset_format="dataframe",
    )

    y_raw = pd.Series(y_raw, name=target_name)
    assert_no_missing(candidate_id, X_raw, y_raw)

    feature_names = selected_feature_names(candidate_id, X_raw)
    X_selected = coerce_numeric_features(candidate_id, X_raw[feature_names])
    assert_no_missing(candidate_id, X_selected, y_raw)

    y_encoded, target_metadata = encode_binary_target(candidate_id, y_raw)
    class_counts = pd.Series(y_encoded).value_counts().sort_index().to_dict()

    loaded[candidate_id] = {
        "did": did,
        "dataset_name": getattr(dataset, "name", ""),
        "target_name": target_name,
        "X": X_selected,
        "y": y_encoded,
        "feature_names": feature_names,
        "target_metadata": target_metadata,
        "class_counts": class_counts,
    }

    warning_rows.append({
        "severity": "info",
        "candidate_id": candidate_id,
        "object": "positive_class_assumption",
        "warning_ru": f"Для бинарных метрик положительным классом принят класс с кодом 1: {target_metadata['positive_class_assumption']!r}. Это диагностическое допущение, а не содержательная интерпретация.",
    })

    print("Название OpenML:", loaded[candidate_id]["dataset_name"])
    print("Целевая переменная:", target_name)
    print("Размер X:", X_selected.shape)
    print("Признаки:", feature_names)
    print("Классы после кодирования:", target_metadata)
    print("Распределение классов:", class_counts)


Загрузка: openml_electricity_151 | OpenML ID: 151
Название OpenML: electricity
Целевая переменная: class
Размер X: (45312, 5)
Признаки: ['nswprice', 'nswdemand', 'vicprice', 'vicdemand', 'transfer']
Классы после кодирования: {'target_class_0': 'DOWN', 'target_class_1': 'UP', 'positive_class_assumption': 'UP'}
Распределение классов: {0: 26075, 1: 19237}

Загрузка: openml_miniboone_41150 | OpenML ID: 41150
Название OpenML: MiniBooNE
Целевая переменная: signal
Размер X: (130064, 50)
Признаки: ['ParticleID_0', 'ParticleID_1', 'ParticleID_2', 'ParticleID_3', 'ParticleID_4', 'ParticleID_5', 'ParticleID_6', 'ParticleID_7', 'ParticleID_8', 'ParticleID_9', 'ParticleID_10', 'ParticleID_11', 'ParticleID_12', 'ParticleID_13', 'ParticleID_14', 'ParticleID_15', 'ParticleID_16', 'ParticleID_17', 'ParticleID_18', 'ParticleID_19', 'ParticleID_20', 'ParticleID_21', 'ParticleID_22', 'ParticleID_23', 'ParticleID_24', 'ParticleID_25', 'ParticleID_26', 'ParticleID_27', 'ParticleID_28', 'ParticleID_29', 'Pa

In [ ]:
from mlcra.metrics import (
    get_positive_class_probability,
    score_binary_classifier,
)


def build_estimators() -> dict[str, Any]:
    return {
        "dummy_prior": DummyClassifier(strategy="prior"),
        "logistic_regression": Pipeline(steps=[
            ("standard_scaler", StandardScaler()),
            ("logistic_regression", LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)),
        ]),
        "hist_gradient_boosting": HistGradientBoostingClassifier(
            random_state=RANDOM_SEED,
            max_iter=100,
            learning_rate=0.1,
        ),
    }

In [8]:
score_rows: list[dict[str, Any]] = []
split_rows: list[dict[str, Any]] = []

for candidate_id, bundle in loaded.items():
    X = bundle["X"]
    y = bundle["y"]

    splitter = StratifiedShuffleSplit(
        n_splits=1,
        test_size=TEST_SIZE,
        random_state=RANDOM_SEED,
    )
    train_index, test_index = next(splitter.split(X, y))

    X_train = X.iloc[train_index].copy()
    X_test = X.iloc[test_index].copy()
    y_train = y[train_index]
    y_test = y[test_index]

    split_rows.append({
        "candidate_id": candidate_id,
        "split_protocol_id": "stratified_random",
        "random_seed": RANDOM_SEED,
        "test_size": TEST_SIZE,
        "n_train": len(train_index),
        "n_test": len(test_index),
        "train_positive_share": float(np.mean(y_train)),
        "test_positive_share": float(np.mean(y_test)),
    })

    print("\n" + "=" * 110)
    print(f"Оценка моделей: {candidate_id}")
    print(f"train={len(train_index)}, test={len(test_index)}")

    for model_id, estimator_template in build_estimators().items():
        estimator = clone(estimator_template)

        fit_start = time.perf_counter()
        estimator.fit(X_train, y_train)
        fit_seconds = time.perf_counter() - fit_start

        score_start = time.perf_counter()
        y_pred = estimator.predict(X_test)
        y_probability = get_positive_class_probability(estimator, X_test)
        metric_values = score_binary_classifier(y_test, y_pred, y_probability)
        score_seconds = time.perf_counter() - score_start

        row = {
            "candidate_id": candidate_id,
            "openml_dataset_id": bundle["did"],
            "dataset_name": bundle["dataset_name"],
            "target_name": bundle["target_name"],
            "target_class_0": bundle["target_metadata"]["target_class_0"],
            "target_class_1": bundle["target_metadata"]["target_class_1"],
            "positive_class_assumption": bundle["target_metadata"]["positive_class_assumption"],
            "feature_count": len(bundle["feature_names"]),
            "feature_names": "; ".join(bundle["feature_names"]),
            "split_protocol_id": "stratified_random",
            "random_seed": RANDOM_SEED,
            "test_size": TEST_SIZE,
            "n_train": len(train_index),
            "n_test": len(test_index),
            "model_id": model_id,
            "fit_seconds": round(fit_seconds, 6),
            "score_seconds": round(score_seconds, 6),
            "row_status": "diagnostic_draft",
            "interpretation_allowed": "no",
        }
        row.update(metric_values)
        score_rows.append(row)

        print(f"{model_id}: fit_seconds={fit_seconds:.3f}, roc_auc={metric_values['roc_auc']:.6f}, pr_auc={metric_values['pr_auc']:.6f}")

scores_df = pd.DataFrame(score_rows)
splits_df = pd.DataFrame(split_rows)
warnings_df = pd.DataFrame(warning_rows)

print("\nСводка разбиений:")
display(splits_df)

print("\nДиагностические результаты. Не интерпретировать как выбор лучшей модели:")
display(scores_df)

print("\nПредупреждения и диагностические допущения:")
display(warnings_df)


Оценка моделей: openml_electricity_151
train=36249, test=9063
dummy_prior: fit_seconds=0.001, roc_auc=0.500000, pr_auc=0.424583
logistic_regression: fit_seconds=0.051, roc_auc=0.810849, pr_auc=0.789111
hist_gradient_boosting: fit_seconds=3.127, roc_auc=0.869187, pr_auc=0.842786

Оценка моделей: openml_miniboone_41150
train=104051, test=26013
dummy_prior: fit_seconds=0.006, roc_auc=0.500000, pr_auc=0.280629
logistic_regression: fit_seconds=3.160, roc_auc=0.937204, pr_auc=0.863241
hist_gradient_boosting: fit_seconds=6.504, roc_auc=0.982782, pr_auc=0.952821

Оценка моделей: openml_click_prediction_small_41434
train=31958, test=7990
dummy_prior: fit_seconds=0.003, roc_auc=0.500000, pr_auc=0.168461
logistic_regression: fit_seconds=0.043, roc_auc=0.609511, pr_auc=0.227142
hist_gradient_boosting: fit_seconds=0.352, roc_auc=0.626839, pr_auc=0.245801

Сводка разбиений:


,candidate_id,split_protocol_id,random_seed,test_size,n_train,n_test,train_positive_share,test_positive_share
0,openml_electricity_151,stratified_random,20260507,0.2,36249,9063,0.424536,0.424583
1,openml_miniboone_41150,stratified_random,20260507,0.2,104051,26013,0.280622,0.280629
2,openml_click_prediction_small_41434,stratified_random,20260507,0.2,31958,7990,0.168409,0.168461



Диагностические результаты. Не интерпретировать как выбор лучшей модели:


,candidate_id,openml_dataset_id,dataset_name,target_name,target_class_0,target_class_1,positive_class_assumption,feature_count,feature_names,split_protocol_id,random_seed,test_size,n_train,n_test,model_id,fit_seconds,score_seconds,row_status,interpretation_allowed,roc_auc,pr_auc,f1,balanced_accuracy,log_loss,brier_score
0,openml_electricity_151,151,electricity,class,DOWN,UP,UP,5,nswprice; nswdemand; vicprice; vicdemand; tran...,stratified_random,20260507,0.2,36249,9063,dummy_prior,0.001495,0.022945,diagnostic_draft,no,0.500000,0.424583,0.000000,0.500000,0.681728,0.244312
1,openml_electricity_151,151,electricity,class,DOWN,UP,UP,5,nswprice; nswdemand; vicprice; vicdemand; tran...,stratified_random,20260507,0.2,36249,9063,logistic_regression,0.050800,0.026100,diagnostic_draft,no,0.810849,0.789111,0.682326,0.739013,0.510277,0.169147
2,openml_electricity_151,151,electricity,class,DOWN,UP,UP,5,nswprice; nswdemand; vicprice; vicdemand; tran...,stratified_random,20260507,0.2,36249,9063,hist_gradient_boosting,3.126866,0.126004,diagnostic_draft,no,0.869187,0.842786,0.731335,0.774849,0.440709,0.144312
3,openml_miniboone_41150,41150,MiniBooNE,signal,False,True,True,50,ParticleID_0; ParticleID_1; ParticleID_2; Part...,stratified_random,20260507,0.2,104051,26013,dummy_prior,0.005551,0.028307,diagnostic_draft,no,0.500000,0.280629,0.000000,0.500000,0.593546,0.201876
4,openml_miniboone_41150,41150,MiniBooNE,signal,False,True,True,50,ParticleID_0; ParticleID_1; ParticleID_2; Part...,stratified_random,20260507,0.2,104051,26013,logistic_regression,3.160381,0.074018,diagnostic_draft,no,0.937204,0.863241,0.777859,0.835331,0.288947,0.086238
5,openml_miniboone_41150,41150,MiniBooNE,signal,False,True,True,50,ParticleID_0; ParticleID_1; ParticleID_2; Part...,stratified_random,20260507,0.2,104051,26013,hist_gradient_boosting,6.503891,0.752374,diagnostic_draft,no,0.982782,0.952821,0.893171,0.927123,0.150734,0.044404
6,openml_click_prediction_small_41434,41434,Click_prediction_small,click,0,1,1,3,impression; depth; position,stratified_random,20260507,0.2,31958,7990,dummy_prior,0.002711,0.017144,diagnostic_draft,no,0.500000,0.168461,0.000000,0.500000,0.453437,0.140082
7,openml_click_prediction_small_41434,41434,Click_prediction_small,click,0,1,1,3,impression; depth; position,stratified_random,20260507,0.2,31958,7990,logistic_regression,0.042569,0.021200,diagnostic_draft,no,0.609511,0.227142,0.011765,0.502520,0.443156,0.136852
8,openml_click_prediction_small_41434,41434,Click_prediction_small,click,0,1,1,3,impression; depth; position,stratified_random,20260507,0.2,31958,7990,hist_gradient_boosting,0.351577,0.055322,diagnostic_draft,no,0.626839,0.245801,0.033165,0.507189,0.437173,0.135419



Предупреждения и диагностические допущения:


,severity,candidate_id,object,warning_ru
0,info,openml_electricity_151,positive_class_assumption,Для бинарных метрик положительным классом прин...
1,info,openml_miniboone_41150,positive_class_assumption,Для бинарных метрик положительным классом прин...
2,info,openml_click_prediction_small_41434,positive_class_assumption,Для бинарных метрик положительным классом прин...


In [9]:
environment_rows = []
for package_name in ["python", "platform", "openml", "sklearn", "pandas", "numpy"]:
    if package_name == "python":
        version = sys.version.replace("\n", " ")
    elif package_name == "platform":
        version = platform.platform()
    else:
        module_name = "sklearn" if package_name == "sklearn" else package_name
        module = importlib.import_module(module_name)
        version = getattr(module, "__version__", "unknown")
    environment_rows.append({"name": package_name, "version": version})

environment_df = pd.DataFrame(environment_rows)

scores_df.to_csv(SCORES_PATH, index=False, encoding="utf-8-sig")
warnings_df.to_csv(WARNINGS_PATH, index=False, encoding="utf-8-sig")
environment_df.to_csv(ENVIRONMENT_PATH, index=False, encoding="utf-8-sig")

print("Сохранены диагностические файлы:")
print(SCORES_PATH.relative_to(PROJECT_ROOT))
print(WARNINGS_PATH.relative_to(PROJECT_ROOT))
print(ENVIRONMENT_PATH.relative_to(PROJECT_ROOT))

print("\nКонтроль границ:")
print("1. dataset_candidate_registry.csv не изменялся этой тетрадью.")
print("2. Статусы кандидатов не менялись.")
print("3. Победитель не выбирался.")
print("4. Результаты имеют статус diagnostic_draft.")

Сохранены диагностические файлы:
data_registry\openml_first_diagnostic_scores.csv
data_registry\openml_first_diagnostic_warnings.csv
data_registry\openml_first_diagnostic_environment.csv

Контроль границ:
1. dataset_candidate_registry.csv не изменялся этой тетрадью.
2. Статусы кандидатов не менялись.
3. Победитель не выбирался.
4. Результаты имеют статус diagnostic_draft.


# 04B — Второй диагностический контур OpenML-кандидатов

Цель: выполнить проверку чувствительности первого диагностического результата к способу разбиения, зерну случайности и политике обработки идентификаторных признаков.

Контуры:

1. `openml_electricity_151`: временное удержанное разбиение — обучение на ранней части последовательности, проверка на поздней части последовательности;
2. `openml_miniboone_41150`: повторные стратифицированные случайные разбиения на нескольких зернах случайности;
3. `openml_click_prediction_small_41434`: безопасный контур без идентификаторов против ограниченного рискованного контура с идентификаторами.

Граница интерпретации: результаты второго контура сохраняются как `diagnostic_draft`; они не меняют статусы кандидатов и не выбирают победителя.

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

SECOND_RANDOM_SEEDS = [20260507, 20260508, 20260509, 20260510, 20260511]
SECOND_TEST_SIZE = 0.20

SECOND_SCORES_PATH = DATA_REGISTRY_DIR / "openml_second_diagnostic_scores.csv"
SECOND_SUMMARY_PATH = DATA_REGISTRY_DIR / "openml_second_diagnostic_summary.csv"
SECOND_WARNINGS_PATH = DATA_REGISTRY_DIR / "openml_second_diagnostic_warnings.csv"

print("Зерна случайности второго контура:", SECOND_RANDOM_SEEDS)
print("Доля тестовой части:", SECOND_TEST_SIZE)

Зерна случайности второго контура: [20260507, 20260508, 20260509, 20260510, 20260511]
Доля тестовой части: 0.2


In [11]:
def get_candidate_row(candidate_id: str) -> pd.Series:
    rows = candidates[candidates["candidate_id"].eq(candidate_id)]
    if len(rows) != 1:
        raise ValueError(f"Ожидалась одна строка кандидата {candidate_id}, найдено: {len(rows)}")
    return rows.iloc[0]


def load_raw_candidate(candidate_id: str) -> dict[str, Any]:
    row = get_candidate_row(candidate_id)
    did = int(row["source_dataset_id"])
    expected_target = row["target_name"]

    dataset = openml.datasets.get_dataset(did)
    target_name = getattr(dataset, "default_target_attribute", None) or expected_target
    X_raw, y_raw, categorical_indicator, attribute_names = dataset.get_data(
        target=target_name,
        dataset_format="dataframe",
    )

    y_raw = pd.Series(y_raw, name=target_name)
    assert_no_missing(candidate_id, X_raw, y_raw)
    y_encoded, target_metadata = encode_binary_target(candidate_id, y_raw)

    return {
        "candidate_id": candidate_id,
        "did": did,
        "dataset_name": getattr(dataset, "name", ""),
        "target_name": target_name,
        "X_raw": X_raw,
        "y": y_encoded,
        "target_metadata": target_metadata,
    }


def build_selected_numeric_frame(candidate_id: str, X_raw: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    feature_names = selected_feature_names(candidate_id, X_raw)
    X_selected = coerce_numeric_features(candidate_id, X_raw[feature_names])
    return X_selected, feature_names


def make_stratified_indices(X: pd.DataFrame, y: np.ndarray, seed: int) -> tuple[np.ndarray, np.ndarray]:
    splitter = StratifiedShuffleSplit(
        n_splits=1,
        test_size=SECOND_TEST_SIZE,
        random_state=seed,
    )
    return next(splitter.split(X, y))


def make_electricity_temporal_indices(X_raw: pd.DataFrame, y: np.ndarray) -> tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    required_order_columns = ["date", "day", "period"]
    missing = [col for col in required_order_columns if col not in X_raw.columns]
    if missing:
        raise KeyError(f"Для временного разбиения electricity отсутствуют служебные поля: {missing}")

    order_frame = pd.DataFrame(index=X_raw.index)

    parsed_date = pd.to_datetime(X_raw["date"], errors="coerce")
    if parsed_date.notna().all():
        order_frame["date_order"] = parsed_date
    else:
        order_frame["date_order"] = pd.Categorical(X_raw["date"].astype(str)).codes

    for column in ["day", "period"]:
        numeric_values = pd.to_numeric(X_raw[column], errors="coerce")
        if numeric_values.notna().all():
            order_frame[f"{column}_order"] = numeric_values
        else:
            order_frame[f"{column}_order"] = pd.Categorical(X_raw[column].astype(str)).codes

    ordered_index = order_frame.sort_values(
        ["date_order", "day_order", "period_order"],
        kind="mergesort",
    ).index.to_numpy()

    split_at = int(len(ordered_index) * (1.0 - SECOND_TEST_SIZE))
    train_index = ordered_index[:split_at]
    test_index = ordered_index[split_at:]

    if len(np.unique(y[train_index])) != 2 or len(np.unique(y[test_index])) != 2:
        raise ValueError(
            "Временное разбиение electricity привело к отсутствию одного из классов "
            "в обучающей или тестовой части. Требуется отдельный протокол разбиения."
        )

    split_diagnostics = pd.DataFrame([
        {
            "candidate_id": "openml_electricity_151",
            "split_protocol_id": "temporal_holdout",
            "n_train": len(train_index),
            "n_test": len(test_index),
            "train_positive_share": float(np.mean(y[train_index])),
            "test_positive_share": float(np.mean(y[test_index])),
            "order_columns": "; ".join(required_order_columns),
        }
    ])
    return train_index, test_index, split_diagnostics


def evaluate_one_split(
    candidate_bundle: dict[str, Any],
    X: pd.DataFrame,
    y: np.ndarray,
    train_index: np.ndarray,
    test_index: np.ndarray,
    model_id: str,
    estimator_template: Any,
    split_protocol_id: str,
    contour_id: str,
    feature_policy_id: str,
    seed: int | str,
    feature_names: list[str],
) -> dict[str, Any]:
    estimator = clone(estimator_template)

    X_train = X.iloc[train_index].copy()
    X_test = X.iloc[test_index].copy()
    y_train = y[train_index]
    y_test = y[test_index]

    fit_start = time.perf_counter()
    estimator.fit(X_train, y_train)
    fit_seconds = time.perf_counter() - fit_start

    score_start = time.perf_counter()
    y_pred = estimator.predict(X_test)
    y_probability = get_positive_class_probability(estimator, X_test)
    metric_values = score_binary_classifier(y_test, y_pred, y_probability)
    score_seconds = time.perf_counter() - score_start

    row = {
        "candidate_id": candidate_bundle["candidate_id"],
        "openml_dataset_id": candidate_bundle["did"],
        "dataset_name": candidate_bundle["dataset_name"],
        "target_name": candidate_bundle["target_name"],
        "target_class_0": candidate_bundle["target_metadata"]["target_class_0"],
        "target_class_1": candidate_bundle["target_metadata"]["target_class_1"],
        "positive_class_assumption": candidate_bundle["target_metadata"]["positive_class_assumption"],
        "contour_id": contour_id,
        "feature_policy_id": feature_policy_id,
        "feature_count": len(feature_names),
        "feature_names": "; ".join(feature_names),
        "split_protocol_id": split_protocol_id,
        "random_seed": seed,
        "test_size": SECOND_TEST_SIZE,
        "n_train": len(train_index),
        "n_test": len(test_index),
        "train_positive_share": float(np.mean(y_train)),
        "test_positive_share": float(np.mean(y_test)),
        "model_id": model_id,
        "fit_seconds": round(fit_seconds, 6),
        "score_seconds": round(score_seconds, 6),
        "row_status": "diagnostic_draft",
        "interpretation_allowed": "no",
    }
    row.update(metric_values)
    return row

In [12]:
second_score_rows: list[dict[str, Any]] = []
second_warning_rows: list[dict[str, Any]] = []
second_split_diagnostics: list[pd.DataFrame] = []

# 1. electricity: временное разбиение.
electricity_bundle = load_raw_candidate("openml_electricity_151")
electricity_X, electricity_features = build_selected_numeric_frame(
    "openml_electricity_151",
    electricity_bundle["X_raw"],
)
electricity_y = electricity_bundle["y"]

electricity_train_index, electricity_test_index, electricity_split_diag = make_electricity_temporal_indices(
    electricity_bundle["X_raw"],
    electricity_y,
)
second_split_diagnostics.append(electricity_split_diag)

for model_id, estimator_template in build_estimators().items():
    row = evaluate_one_split(
        candidate_bundle=electricity_bundle,
        X=electricity_X,
        y=electricity_y,
        train_index=electricity_train_index,
        test_index=electricity_test_index,
        model_id=model_id,
        estimator_template=estimator_template,
        split_protocol_id="temporal_holdout",
        contour_id="electricity_temporal_order_split",
        feature_policy_id="safe_numeric_no_time_predictors",
        seed="not_applicable_temporal_order",
        feature_names=electricity_features,
    )
    second_score_rows.append(row)
    print(
        "electricity temporal |",
        model_id,
        "roc_auc=",
        f"{row['roc_auc']:.6f}",
        "pr_auc=",
        f"{row['pr_auc']:.6f}",
    )

second_warning_rows.append({
    "severity": "info",
    "candidate_id": "openml_electricity_151",
    "object": "temporal_order",
    "warning_ru": "Временное разбиение использует date/day/period только для порядка строк; эти поля не включены в модельные признаки.",
})

electricity temporal | dummy_prior roc_auc= 0.500000 pr_auc= 0.410239
electricity temporal | logistic_regression roc_auc= 0.832076 pr_auc= 0.776824
electricity temporal | hist_gradient_boosting roc_auc= 0.838688 pr_auc= 0.781942


In [13]:
# 2. MiniBooNE: повторные стратифицированные случайные разбиения.
miniboone_bundle = load_raw_candidate("openml_miniboone_41150")
miniboone_X, miniboone_features = build_selected_numeric_frame(
    "openml_miniboone_41150",
    miniboone_bundle["X_raw"],
)
miniboone_y = miniboone_bundle["y"]

for seed in SECOND_RANDOM_SEEDS:
    train_index, test_index = make_stratified_indices(miniboone_X, miniboone_y, seed)
    for model_id, estimator_template in build_estimators().items():
        row = evaluate_one_split(
            candidate_bundle=miniboone_bundle,
            X=miniboone_X,
            y=miniboone_y,
            train_index=train_index,
            test_index=test_index,
            model_id=model_id,
            estimator_template=estimator_template,
            split_protocol_id="repeated_stratified_random",
            contour_id="miniboone_repeated_seed_stability",
            feature_policy_id="numeric_predictors",
            seed=seed,
            feature_names=miniboone_features,
        )
        second_score_rows.append(row)
        print(
            "miniboone repeated |",
            seed,
            "|",
            model_id,
            "roc_auc=",
            f"{row['roc_auc']:.6f}",
        )

second_warning_rows.append({
    "severity": "info",
    "candidate_id": "openml_miniboone_41150",
    "object": "repeated_stratified_random",
    "warning_ru": "Повторные разбиения проверяют устойчивость к зерну случайности, но не проверяют временной или групповой дрейф.",
})

miniboone repeated | 20260507 | dummy_prior roc_auc= 0.500000
miniboone repeated | 20260507 | logistic_regression roc_auc= 0.937204
miniboone repeated | 20260507 | hist_gradient_boosting roc_auc= 0.982782
miniboone repeated | 20260508 | dummy_prior roc_auc= 0.500000
miniboone repeated | 20260508 | logistic_regression roc_auc= 0.940482
miniboone repeated | 20260508 | hist_gradient_boosting roc_auc= 0.984204
miniboone repeated | 20260509 | dummy_prior roc_auc= 0.500000
miniboone repeated | 20260509 | logistic_regression roc_auc= 0.938157
miniboone repeated | 20260509 | hist_gradient_boosting roc_auc= 0.982926
miniboone repeated | 20260510 | dummy_prior roc_auc= 0.500000
miniboone repeated | 20260510 | logistic_regression roc_auc= 0.939090
miniboone repeated | 20260510 | hist_gradient_boosting roc_auc= 0.984100
miniboone repeated | 20260511 | dummy_prior roc_auc= 0.500000
miniboone repeated | 20260511 | logistic_regression roc_auc= 0.938783
miniboone repeated | 20260511 | hist_gradient_bo

In [14]:
# 3. Click_prediction_small: безопасный контур против ограниченного рискованного контура с идентификаторами.
def get_click_identifier_features() -> list[str]:
    rows = feature_plan[
        (feature_plan["candidate_id"].eq("openml_click_prediction_small_41434"))
        & (feature_plan["feature_group_id"].eq("click_identifier_features_drop_primary"))
    ]
    if len(rows) != 1:
        raise ValueError("Не найдена ровно одна строка политики идентификаторов Click_prediction_small")
    return split_semicolon_list(rows.iloc[0]["feature_names"])


def build_click_identifier_risk_estimator(numeric_features: list[str], identifier_features: list[str]) -> Pipeline:
    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", StandardScaler(), numeric_features),
            (
                "identifier_limited_one_hot",
                OneHotEncoder(handle_unknown="ignore", max_categories=50, sparse_output=True),
                identifier_features,
            ),
        ],
        remainder="drop",
    )
    return Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("logistic_regression", LogisticRegression(max_iter=1000, random_state=RANDOM_SEED, solver="saga")),
    ])


click_bundle = load_raw_candidate("openml_click_prediction_small_41434")
click_X_raw = click_bundle["X_raw"]
click_y = click_bundle["y"]
click_safe_X, click_safe_features = build_selected_numeric_frame(
    "openml_click_prediction_small_41434",
    click_X_raw,
)
click_identifier_features = get_click_identifier_features()
missing_click_identifiers = [col for col in click_identifier_features if col not in click_X_raw.columns]
if missing_click_identifiers:
    raise KeyError(f"В Click_prediction_small отсутствуют ожидаемые идентификаторные признаки: {missing_click_identifiers}")

click_risky_features = click_safe_features + click_identifier_features
click_risky_X = click_X_raw[click_risky_features].copy()
for column in click_safe_features:
    click_risky_X[column] = pd.to_numeric(click_risky_X[column], errors="raise")
for column in click_identifier_features:
    click_risky_X[column] = click_risky_X[column].astype("string").fillna("__MISSING__")

click_train_index, click_test_index = make_stratified_indices(click_safe_X, click_y, RANDOM_SEED)

click_estimators = {
    "logistic_regression_safe_numeric": Pipeline(steps=[
        ("standard_scaler", StandardScaler()),
        ("logistic_regression", LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)),
    ]),
    "logistic_regression_identifier_risk_limited": build_click_identifier_risk_estimator(
        numeric_features=click_safe_features,
        identifier_features=click_identifier_features,
    ),
}

for model_id, estimator_template in click_estimators.items():
    if model_id.endswith("safe_numeric"):
        X_for_model = click_safe_X
        feature_names = click_safe_features
        feature_policy_id = "safe_numeric_without_identifiers"
    else:
        X_for_model = click_risky_X
        feature_names = click_risky_features
        feature_policy_id = "risk_limited_identifier_one_hot_max_categories_50"

    row = evaluate_one_split(
        candidate_bundle=click_bundle,
        X=X_for_model,
        y=click_y,
        train_index=click_train_index,
        test_index=click_test_index,
        model_id=model_id,
        estimator_template=estimator_template,
        split_protocol_id="stratified_random_identifier_policy_comparison",
        contour_id="click_identifier_ablation",
        feature_policy_id=feature_policy_id,
        seed=RANDOM_SEED,
        feature_names=feature_names,
    )
    second_score_rows.append(row)
    print(
        "click identifier ablation |",
        model_id,
        "roc_auc=",
        f"{row['roc_auc']:.6f}",
        "pr_auc=",
        f"{row['pr_auc']:.6f}",
    )

second_warning_rows.append({
    "severity": "warning",
    "candidate_id": "openml_click_prediction_small_41434",
    "object": "identifier_ablation",
    "warning_ru": "Контур с идентификаторами имеет статус leakage-risk: его нельзя использовать как финальную оценку качества модели.",
})
second_warning_rows.append({
    "severity": "info",
    "candidate_id": "openml_click_prediction_small_41434",
    "object": "identifier_encoding_limit",
    "warning_ru": "Идентификаторы кодируются ограниченным прямым кодированием с max_categories=50; это диагностический стресс-тест, а не утвержденная финальная предобработка.",
})

click identifier ablation | logistic_regression_safe_numeric roc_auc= 0.609511 pr_auc= 0.227142


c:\Users\Vanargo\Desktop\ML-CRA\ml-cra-venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


click identifier ablation | logistic_regression_identifier_risk_limited roc_auc= 0.632015 pr_auc= 0.260422


In [15]:
second_scores_df = pd.DataFrame(second_score_rows)
second_warnings_df = pd.DataFrame(second_warning_rows)
second_split_diagnostics_df = pd.concat(second_split_diagnostics, ignore_index=True) if second_split_diagnostics else pd.DataFrame()

metric_columns = ["roc_auc", "pr_auc", "f1", "balanced_accuracy", "log_loss", "brier_score"]
summary_rows = []
for keys, group in second_scores_df.groupby(["candidate_id", "contour_id", "feature_policy_id", "model_id"], dropna=False):
    candidate_id, contour_id, feature_policy_id, model_id = keys
    row = {
        "candidate_id": candidate_id,
        "contour_id": contour_id,
        "feature_policy_id": feature_policy_id,
        "model_id": model_id,
        "n_runs": int(len(group)),
        "row_status": "diagnostic_draft",
        "interpretation_allowed": "no",
    }
    for metric in metric_columns:
        row[f"{metric}_mean"] = float(group[metric].mean())
        row[f"{metric}_std"] = float(group[metric].std(ddof=0))
        row[f"{metric}_min"] = float(group[metric].min())
        row[f"{metric}_max"] = float(group[metric].max())
    summary_rows.append(row)

second_summary_df = pd.DataFrame(summary_rows).sort_values(
    ["candidate_id", "contour_id", "model_id"],
    kind="mergesort",
)

second_scores_df.to_csv(SECOND_SCORES_PATH, index=False, encoding="utf-8-sig")
second_summary_df.to_csv(SECOND_SUMMARY_PATH, index=False, encoding="utf-8-sig")
second_warnings_df.to_csv(SECOND_WARNINGS_PATH, index=False, encoding="utf-8-sig")

print("Сохранены файлы второго диагностического контура:")
print(SECOND_SCORES_PATH.relative_to(PROJECT_ROOT))
print(SECOND_SUMMARY_PATH.relative_to(PROJECT_ROOT))
print(SECOND_WARNINGS_PATH.relative_to(PROJECT_ROOT))

print("\nДиагностика временного разбиения electricity:")
display(second_split_diagnostics_df)

print("\nСводка второго диагностического контура:")
display(second_summary_df)

print("\nПредупреждения второго контура:")
display(second_warnings_df)

print("\nКонтроль границ:")
print("1. dataset_candidate_registry.csv не изменялся этой тетрадью.")
print("2. Статусы кандидатов не менялись.")
print("3. Контур Click с идентификаторами имеет статус leakage-risk и не является финальной оценкой.")
print("4. Все строки второго контура имеют статус diagnostic_draft.")

Сохранены файлы второго диагностического контура:
data_registry\openml_second_diagnostic_scores.csv
data_registry\openml_second_diagnostic_summary.csv
data_registry\openml_second_diagnostic_warnings.csv

Диагностика временного разбиения electricity:


,candidate_id,split_protocol_id,n_train,n_test,train_positive_share,test_positive_share,order_columns
0,openml_electricity_151,temporal_holdout,36249,9063,0.428122,0.410239,date; day; period



Сводка второго диагностического контура:


,candidate_id,contour_id,feature_policy_id,model_id,n_runs,row_status,interpretation_allowed,roc_auc_mean,roc_auc_std,roc_auc_min,roc_auc_max,pr_auc_mean,pr_auc_std,pr_auc_min,pr_auc_max,f1_mean,f1_std,f1_min,f1_max,balanced_accuracy_mean,balanced_accuracy_std,balanced_accuracy_min,balanced_accuracy_max,log_loss_mean,log_loss_std,log_loss_min,log_loss_max,brier_score_mean,brier_score_std,brier_score_min,brier_score_max
0,openml_click_prediction_small_41434,click_identifier_ablation,risk_limited_identifier_one_hot_max_categories_50,logistic_regression_identifier_risk_limited,1,diagnostic_draft,no,0.632015,0.000000,0.632015,0.632015,0.260422,0.000000,0.260422,0.260422,0.236036,0.000000,0.236036,0.236036,0.551269,0.000000,0.551269,0.551269,0.505277,0.000000,0.505277,0.505277,0.164479,0.000000e+00,0.164479,0.164479
1,openml_click_prediction_small_41434,click_identifier_ablation,safe_numeric_without_identifiers,logistic_regression_safe_numeric,1,diagnostic_draft,no,0.609511,0.000000,0.609511,0.609511,0.227142,0.000000,0.227142,0.227142,0.011765,0.000000,0.011765,0.011765,0.502520,0.000000,0.502520,0.502520,0.443156,0.000000,0.443156,0.443156,0.136852,0.000000e+00,0.136852,0.136852
2,openml_electricity_151,electricity_temporal_order_split,safe_numeric_no_time_predictors,dummy_prior,1,diagnostic_draft,no,0.500000,0.000000,0.500000,0.500000,0.410239,0.000000,0.410239,0.410239,0.000000,0.000000,0.000000,0.000000,0.500000,0.000000,0.500000,0.500000,0.677601,0.000000,0.677601,0.677601,0.242263,0.000000e+00,0.242263,0.242263
3,openml_electricity_151,electricity_temporal_order_split,safe_numeric_no_time_predictors,hist_gradient_boosting,1,diagnostic_draft,no,0.838688,0.000000,0.838688,0.838688,0.781942,0.000000,0.781942,0.781942,0.659060,0.000000,0.659060,0.659060,0.726833,0.000000,0.726833,0.726833,0.480850,0.000000,0.480850,0.480850,0.160537,0.000000e+00,0.160537,0.160537
4,openml_electricity_151,electricity_temporal_order_split,safe_numeric_no_time_predictors,logistic_regression,1,diagnostic_draft,no,0.832076,0.000000,0.832076,0.832076,0.776824,0.000000,0.776824,0.776824,0.636643,0.000000,0.636643,0.636643,0.718418,0.000000,0.718418,0.718418,0.510664,0.000000,0.510664,0.510664,0.167291,0.000000e+00,0.167291,0.167291
5,openml_miniboone_41150,miniboone_repeated_seed_stability,numeric_predictors,dummy_prior,5,diagnostic_draft,no,0.500000,0.000000,0.500000,0.500000,0.280629,0.000000,0.280629,0.280629,0.000000,0.000000,0.000000,0.000000,0.500000,0.000000,0.500000,0.500000,0.593546,0.000000,0.593546,0.593546,0.201876,2.775558e-17,0.201876,0.201876
6,openml_miniboone_41150,miniboone_repeated_seed_stability,numeric_predictors,hist_gradient_boosting,5,diagnostic_draft,no,0.983444,0.000595,0.982782,0.984204,0.955148,0.001541,0.952821,0.956985,0.895585,0.001613,0.893171,0.897700,0.928877,0.001027,0.927123,0.929745,0.147972,0.002306,0.145055,0.150734,0.043504,7.552646e-04,0.042610,0.044404
7,openml_miniboone_41150,miniboone_repeated_seed_stability,numeric_predictors,logistic_regression,5,diagnostic_draft,no,0.938743,0.001082,0.937204,0.940482,0.866641,0.002719,0.863241,0.870505,0.776090,0.002457,0.771344,0.778024,0.833514,0.002057,0.829771,0.835331,0.286073,0.002270,0.282181,0.288947,0.085677,5.814236e-04,0.084723,0.086291



Предупреждения второго контура:


,severity,candidate_id,object,warning_ru
0,info,openml_electricity_151,temporal_order,Временное разбиение использует date/day/period...
1,info,openml_miniboone_41150,repeated_stratified_random,Повторные разбиения проверяют устойчивость к з...
2,warning,openml_click_prediction_small_41434,identifier_ablation,Контур с идентификаторами имеет статус leakage...
3,info,openml_click_prediction_small_41434,identifier_encoding_limit,Идентификаторы кодируются ограниченным прямым ...



Контроль границ:
1. dataset_candidate_registry.csv не изменялся этой тетрадью.
2. Статусы кандидатов не менялись.
3. Контур Click с идентификаторами имеет статус leakage-risk и не является финальной оценкой.
4. Все строки второго контура имеют статус diagnostic_draft.


# 04C — Минимальный исследовательский контур MiniBooNE

Цель: выполнить минимальную исследовательскую оценку основного табличного кандидата `openml_miniboone_41150` по протоколу `REPORT_32`.

Контур:

1. данные: OpenML ID `41150`, целевая переменная `signal`, признаки `ParticleID_0` ... `ParticleID_49`;
2. разбиение: `RepeatedStratifiedKFold` — повторная стратифицированная k-блочная проверка, `n_splits=5`, `n_repeats=2`, `random_state=20260507`;
3. модели: `dummy_prior`, `logistic_regression`, `hist_gradient_boosting`;
4. статус строк: `research_draft`;
5. граница интерпретации: подбор гиперпараметров запрещен; результаты не являются финальной оценкой ML-CRA.

In [16]:
from sklearn.model_selection import RepeatedStratifiedKFold

MINIBOONE_RESEARCH_PROTOCOL_ID = "report33_miniboone_minimal_research"
MINIBOONE_RESEARCH_CANDIDATE_ID = "openml_miniboone_41150"
MINIBOONE_RESEARCH_N_SPLITS = 5
MINIBOONE_RESEARCH_N_REPEATS = 2
MINIBOONE_RESEARCH_RANDOM_SEED = 20260507

MINIBOONE_RESEARCH_SCORES_PATH = DATA_REGISTRY_DIR / "openml_miniboone_research_scores.csv"
MINIBOONE_RESEARCH_SUMMARY_PATH = DATA_REGISTRY_DIR / "openml_miniboone_research_summary.csv"
MINIBOONE_RESEARCH_WARNINGS_PATH = DATA_REGISTRY_DIR / "openml_miniboone_research_warnings.csv"
MINIBOONE_RESEARCH_ENVIRONMENT_PATH = DATA_REGISTRY_DIR / "openml_miniboone_research_environment.csv"

print("Протокол:", MINIBOONE_RESEARCH_PROTOCOL_ID)
print("Кандидат:", MINIBOONE_RESEARCH_CANDIDATE_ID)
print("RepeatedStratifiedKFold:", {
    "n_splits": MINIBOONE_RESEARCH_N_SPLITS,
    "n_repeats": MINIBOONE_RESEARCH_N_REPEATS,
    "random_state": MINIBOONE_RESEARCH_RANDOM_SEED,
})

Протокол: report33_miniboone_minimal_research
Кандидат: openml_miniboone_41150
RepeatedStratifiedKFold: {'n_splits': 5, 'n_repeats': 2, 'random_state': 20260507}


In [17]:
def model_parameter_text(model_id: str) -> str:
    if model_id == "dummy_prior":
        return "DummyClassifier(strategy='prior')"
    if model_id == "logistic_regression":
        return "Pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=20260507))"
    if model_id == "hist_gradient_boosting":
        return "HistGradientBoostingClassifier(max_iter=100, learning_rate=0.1, random_state=20260507)"
    raise ValueError(f"Неизвестная модель: {model_id}")


def report33_cv_position(split_number: int) -> tuple[int, int]:
    repeat_number = (split_number // MINIBOONE_RESEARCH_N_SPLITS) + 1
    fold_number = (split_number % MINIBOONE_RESEARCH_N_SPLITS) + 1
    return repeat_number, fold_number


def evaluate_miniboone_research_fold(
    candidate_bundle: dict[str, Any],
    X: pd.DataFrame,
    y: np.ndarray,
    train_index: np.ndarray,
    test_index: np.ndarray,
    model_id: str,
    estimator_template: Any,
    split_number: int,
    feature_names: list[str],
) -> tuple[dict[str, Any], list[dict[str, Any]]]:
    repeat_number, fold_number = report33_cv_position(split_number)
    captured_warning_rows: list[dict[str, Any]] = []

    with warnings.catch_warnings(record=True) as captured_warnings:
        warnings.simplefilter("always")
        row = evaluate_one_split(
            candidate_bundle=candidate_bundle,
            X=X,
            y=y,
            train_index=train_index,
            test_index=test_index,
            model_id=model_id,
            estimator_template=estimator_template,
            split_protocol_id="repeated_stratified_kfold",
            contour_id="miniboone_minimal_research_cv",
            feature_policy_id="numeric_particleid_0_49_locked",
            seed=MINIBOONE_RESEARCH_RANDOM_SEED,
            feature_names=feature_names,
        )

    row.update({
        "protocol_id": MINIBOONE_RESEARCH_PROTOCOL_ID,
        "cv_split_number": split_number + 1,
        "cv_repeat_number": repeat_number,
        "cv_fold_number": fold_number,
        "n_splits": MINIBOONE_RESEARCH_N_SPLITS,
        "n_repeats": MINIBOONE_RESEARCH_N_REPEATS,
        "model_parameters": model_parameter_text(model_id),
        "row_status": "research_draft",
        "interpretation_allowed": "limited_protocol_review_only",
        "hyperparameter_tuning_used": "no",
    })

    for captured_warning in captured_warnings:
        captured_warning_rows.append({
            "severity": "warning",
            "candidate_id": MINIBOONE_RESEARCH_CANDIDATE_ID,
            "protocol_id": MINIBOONE_RESEARCH_PROTOCOL_ID,
            "cv_split_number": split_number + 1,
            "cv_repeat_number": repeat_number,
            "cv_fold_number": fold_number,
            "model_id": model_id,
            "object": captured_warning.category.__name__,
            "warning_ru": str(captured_warning.message),
        })

    return row, captured_warning_rows

In [18]:
miniboone_research_bundle = load_raw_candidate(MINIBOONE_RESEARCH_CANDIDATE_ID)
miniboone_research_X, miniboone_research_features = build_selected_numeric_frame(
    MINIBOONE_RESEARCH_CANDIDATE_ID,
    miniboone_research_bundle["X_raw"],
)
miniboone_research_y = miniboone_research_bundle["y"]

if miniboone_research_features != [f"ParticleID_{i}" for i in range(50)]:
    raise ValueError(
        "Нарушен замок признаков MiniBooNE: ожидались ParticleID_0 ... ParticleID_49, "
        f"получено: {miniboone_research_features}"
    )

miniboone_research_class_counts = pd.Series(miniboone_research_y).value_counts().sort_index().to_dict()
print("MiniBooNE X:", miniboone_research_X.shape)
print("MiniBooNE признаки:", len(miniboone_research_features))
print("Распределение классов:", miniboone_research_class_counts)
print("Целевые классы:", miniboone_research_bundle["target_metadata"])

MiniBooNE X: (130064, 50)
MiniBooNE признаки: 50
Распределение классов: {0: 93565, 1: 36499}
Целевые классы: {'target_class_0': 'False', 'target_class_1': 'True', 'positive_class_assumption': 'True'}


In [19]:
miniboone_research_score_rows: list[dict[str, Any]] = []
miniboone_research_warning_rows: list[dict[str, Any]] = []

miniboone_cv = RepeatedStratifiedKFold(
    n_splits=MINIBOONE_RESEARCH_N_SPLITS,
    n_repeats=MINIBOONE_RESEARCH_N_REPEATS,
    random_state=MINIBOONE_RESEARCH_RANDOM_SEED,
)

for split_number, (train_index, test_index) in enumerate(
    miniboone_cv.split(miniboone_research_X, miniboone_research_y)
):
    repeat_number, fold_number = report33_cv_position(split_number)
    print("\n" + "=" * 110)
    print(f"MiniBooNE CV: repeat={repeat_number}, fold={fold_number}")
    print(
        "train/test:",
        len(train_index),
        len(test_index),
        "positive_share:",
        round(float(np.mean(miniboone_research_y[train_index])), 6),
        round(float(np.mean(miniboone_research_y[test_index])), 6),
    )

    for model_id, estimator_template in build_estimators().items():
        row, captured_warning_rows = evaluate_miniboone_research_fold(
            candidate_bundle=miniboone_research_bundle,
            X=miniboone_research_X,
            y=miniboone_research_y,
            train_index=train_index,
            test_index=test_index,
            model_id=model_id,
            estimator_template=estimator_template,
            split_number=split_number,
            feature_names=miniboone_research_features,
        )
        miniboone_research_score_rows.append(row)
        miniboone_research_warning_rows.extend(captured_warning_rows)
        print(
            model_id,
            "roc_auc=", f"{row['roc_auc']:.6f}",
            "pr_auc=", f"{row['pr_auc']:.6f}",
            "f1=", f"{row['f1']:.6f}",
        )

miniboone_research_warning_rows.append({
    "severity": "info",
    "candidate_id": MINIBOONE_RESEARCH_CANDIDATE_ID,
    "protocol_id": MINIBOONE_RESEARCH_PROTOCOL_ID,
    "cv_split_number": "all",
    "cv_repeat_number": "all",
    "cv_fold_number": "all",
    "model_id": "all",
    "object": "hyperparameter_tuning",
    "warning_ru": "Подбор гиперпараметров не выполнялся; результаты допустимы только как минимальная внешняя проверка утвержденного контура.",
})


MiniBooNE CV: repeat=1, fold=1
train/test: 104051 26013 positive_share: 0.280622 0.280629
dummy_prior roc_auc= 0.500000 pr_auc= 0.280629 f1= 0.000000
logistic_regression roc_auc= 0.936721 pr_auc= 0.862086 f1= 0.774730
hist_gradient_boosting roc_auc= 0.982057 pr_auc= 0.952388 f1= 0.888708

MiniBooNE CV: repeat=1, fold=2
train/test: 104051 26013 positive_share: 0.280622 0.280629
dummy_prior roc_auc= 0.500000 pr_auc= 0.280629 f1= 0.000000
logistic_regression roc_auc= 0.940414 pr_auc= 0.868521 f1= 0.773053
hist_gradient_boosting roc_auc= 0.983633 pr_auc= 0.956345 f1= 0.897405

MiniBooNE CV: repeat=1, fold=3
train/test: 104051 26013 positive_share: 0.280622 0.280629
dummy_prior roc_auc= 0.500000 pr_auc= 0.280629 f1= 0.000000
logistic_regression roc_auc= 0.940193 pr_auc= 0.872183 f1= 0.782049
hist_gradient_boosting roc_auc= 0.984596 pr_auc= 0.958674 f1= 0.898964

MiniBooNE CV: repeat=1, fold=4
train/test: 104051 26013 positive_share: 0.280622 0.280629
dummy_prior roc_auc= 0.500000 pr_auc= 0

In [20]:
miniboone_research_scores_df = pd.DataFrame(miniboone_research_score_rows)
miniboone_research_warnings_df = pd.DataFrame(miniboone_research_warning_rows)

expected_rows = MINIBOONE_RESEARCH_N_SPLITS * MINIBOONE_RESEARCH_N_REPEATS * len(build_estimators())
if len(miniboone_research_scores_df) != expected_rows:
    raise ValueError(
        f"Неожиданное число строк результатов: ожидалось {expected_rows}, "
        f"получено {len(miniboone_research_scores_df)}"
    )

metric_columns = ["roc_auc", "pr_auc", "f1", "balanced_accuracy", "log_loss", "brier_score"]
summary_rows = []
for keys, group in miniboone_research_scores_df.groupby(
    ["candidate_id", "protocol_id", "contour_id", "feature_policy_id", "model_id"],
    dropna=False,
):
    candidate_id, protocol_id, contour_id, feature_policy_id, model_id = keys
    row = {
        "candidate_id": candidate_id,
        "protocol_id": protocol_id,
        "contour_id": contour_id,
        "feature_policy_id": feature_policy_id,
        "model_id": model_id,
        "model_parameters": model_parameter_text(model_id),
        "n_external_blocks": int(len(group)),
        "n_splits": MINIBOONE_RESEARCH_N_SPLITS,
        "n_repeats": MINIBOONE_RESEARCH_N_REPEATS,
        "random_state": MINIBOONE_RESEARCH_RANDOM_SEED,
        "row_status": "research_draft",
        "interpretation_allowed": "limited_protocol_review_only",
        "hyperparameter_tuning_used": "no",
    }
    for metric in metric_columns:
        row[f"{metric}_mean"] = float(group[metric].mean())
        row[f"{metric}_std"] = float(group[metric].std(ddof=0))
        row[f"{metric}_min"] = float(group[metric].min())
        row[f"{metric}_max"] = float(group[metric].max())
    summary_rows.append(row)

miniboone_research_summary_df = pd.DataFrame(summary_rows).sort_values(
    ["candidate_id", "model_id"],
    kind="mergesort",
)

print("Строки результатов:", len(miniboone_research_scores_df))
print("Сводка MiniBooNE research:")
display(miniboone_research_summary_df)

print("Предупреждения MiniBooNE research:")
display(miniboone_research_warnings_df)

Строки результатов: 30
Сводка MiniBooNE research:


,candidate_id,protocol_id,contour_id,feature_policy_id,model_id,model_parameters,n_external_blocks,n_splits,n_repeats,random_state,row_status,interpretation_allowed,hyperparameter_tuning_used,roc_auc_mean,roc_auc_std,roc_auc_min,roc_auc_max,pr_auc_mean,pr_auc_std,pr_auc_min,pr_auc_max,f1_mean,f1_std,f1_min,f1_max,balanced_accuracy_mean,balanced_accuracy_std,balanced_accuracy_min,balanced_accuracy_max,log_loss_mean,log_loss_std,log_loss_min,log_loss_max,brier_score_mean,brier_score_std,brier_score_min,brier_score_max
0,openml_miniboone_41150,report33_miniboone_minimal_research,miniboone_minimal_research_cv,numeric_particleid_0_49_locked,dummy_prior,DummyClassifier(strategy='prior'),10,5,2,20260507,research_draft,limited_protocol_review_only,no,0.500000,0.000000,0.500000,0.500000,0.280623,0.000011,0.280601,0.280629,0.000000,0.000000,0.000000,0.000000,0.500000,0.000000,0.500000,0.500000,0.593541,0.000010,0.593520,0.593546,0.201874,0.000005,0.201864,0.201876
1,openml_miniboone_41150,report33_miniboone_minimal_research,miniboone_minimal_research_cv,numeric_particleid_0_49_locked,hist_gradient_boosting,"HistGradientBoostingClassifier(max_iter=100, l...",10,5,2,20260507,research_draft,limited_protocol_review_only,no,0.983477,0.000736,0.982057,0.984667,0.955593,0.001999,0.952388,0.958674,0.895582,0.003101,0.888708,0.900586,0.929308,0.001859,0.924766,0.931969,0.147972,0.002960,0.143141,0.153873,0.043603,0.001051,0.041776,0.045644
2,openml_miniboone_41150,report33_miniboone_minimal_research,miniboone_minimal_research_cv,numeric_particleid_0_49_locked,logistic_regression,"Pipeline(StandardScaler(), LogisticRegression(...",10,5,2,20260507,research_draft,limited_protocol_review_only,no,0.939278,0.001284,0.936721,0.941325,0.867408,0.002849,0.862086,0.872183,0.776990,0.002756,0.772805,0.782049,0.834122,0.001872,0.830738,0.837224,0.285299,0.002487,0.281366,0.290135,0.085489,0.000767,0.084325,0.087380


Предупреждения MiniBooNE research:


,severity,candidate_id,protocol_id,cv_split_number,cv_repeat_number,cv_fold_number,model_id,object,warning_ru
0,info,openml_miniboone_41150,report33_miniboone_minimal_research,all,all,all,all,hyperparameter_tuning,Подбор гиперпараметров не выполнялся; результа...


In [21]:
miniboone_research_environment_rows = []
for package_name in ["python", "platform", "openml", "sklearn", "pandas", "numpy"]:
    if package_name == "python":
        version = sys.version.replace("\n", " ")
    elif package_name == "platform":
        version = platform.platform()
    else:
        module_name = "sklearn" if package_name == "sklearn" else package_name
        module = importlib.import_module(module_name)
        version = getattr(module, "__version__", "unknown")
    miniboone_research_environment_rows.append({
        "name": package_name,
        "version": version,
        "protocol_id": MINIBOONE_RESEARCH_PROTOCOL_ID,
    })

miniboone_research_environment_df = pd.DataFrame(miniboone_research_environment_rows)

miniboone_research_scores_df.to_csv(MINIBOONE_RESEARCH_SCORES_PATH, index=False, encoding="utf-8-sig")
miniboone_research_summary_df.to_csv(MINIBOONE_RESEARCH_SUMMARY_PATH, index=False, encoding="utf-8-sig")
miniboone_research_warnings_df.to_csv(MINIBOONE_RESEARCH_WARNINGS_PATH, index=False, encoding="utf-8-sig")
miniboone_research_environment_df.to_csv(MINIBOONE_RESEARCH_ENVIRONMENT_PATH, index=False, encoding="utf-8-sig")

print("Сохранены файлы минимального исследовательского контура MiniBooNE:")
print(MINIBOONE_RESEARCH_SCORES_PATH.relative_to(PROJECT_ROOT))
print(MINIBOONE_RESEARCH_SUMMARY_PATH.relative_to(PROJECT_ROOT))
print(MINIBOONE_RESEARCH_WARNINGS_PATH.relative_to(PROJECT_ROOT))
print(MINIBOONE_RESEARCH_ENVIRONMENT_PATH.relative_to(PROJECT_ROOT))

print("\nКонтроль границ:")
print("1. Подбор гиперпараметров не выполнялся.")
print("2. dataset_candidate_registry.csv не изменялся этой тетрадью.")
print("3. Статус кандидата не повышался.")
print("4. Результаты имеют статус research_draft.")
print("5. Интерпретация разрешена только для проверки выполнения протокола REPORT_32.")

Сохранены файлы минимального исследовательского контура MiniBooNE:
data_registry\openml_miniboone_research_scores.csv
data_registry\openml_miniboone_research_summary.csv
data_registry\openml_miniboone_research_warnings.csv
data_registry\openml_miniboone_research_environment.csv

Контроль границ:
1. Подбор гиперпараметров не выполнялся.
2. dataset_candidate_registry.csv не изменялся этой тетрадью.
3. Статус кандидата не повышался.
4. Результаты имеют статус research_draft.
5. Интерпретация разрешена только для проверки выполнения протокола REPORT_32.


## 7 — Вложенная проверка MiniBooNE v01

Цель блока: выполнить зафиксированный до запуска вложенный протокол MiniBooNE v01.

Границы интерпретации:

1. внешний контур не участвует в выборе параметров;
2. параметры `hist_gradient_boosting` выбираются только во внутреннем контуре;
3. `logistic_regression` и `dummy_prior` остаются контрольными моделями без настройки параметров;
4. статус выходов — `nested_research_draft`;
5. `dataset_candidate_registry.csv` этим блоком не изменяется.

In [22]:
import hashlib
import json

from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold, StratifiedKFold

MINIBOONE_NESTED_PROTOCOL_ID = "miniboone_nested_cv_v01"
MINIBOONE_NESTED_CANDIDATE_ID = "openml_miniboone_41150"
MINIBOONE_NESTED_N_SPLITS = 5
MINIBOONE_NESTED_N_REPEATS = 2
MINIBOONE_NESTED_INNER_N_SPLITS = 3
MINIBOONE_NESTED_RANDOM_SEED = 20260507

MINIBOONE_NESTED_MODEL_SPACE_PATH = (
    PROJECT_ROOT
    / "configs"
    / "model_spaces"
    / "miniboone_hist_gradient_boosting_nested_space.csv"
)
MINIBOONE_NESTED_PROTOCOL_LOCK_PATH = DATA_REGISTRY_DIR / "openml_miniboone_nested_cv_protocol_lock.csv"
MINIBOONE_NESTED_EXPECTED_SCHEMA_PATH = DATA_REGISTRY_DIR / "openml_miniboone_nested_cv_expected_output_schema.csv"

MINIBOONE_NESTED_OUTER_SCORES_PATH = DATA_REGISTRY_DIR / "openml_miniboone_nested_outer_scores.csv"
MINIBOONE_NESTED_SELECTED_PARAMS_PATH = DATA_REGISTRY_DIR / "openml_miniboone_nested_selected_params.csv"
MINIBOONE_NESTED_SUMMARY_PATH = DATA_REGISTRY_DIR / "openml_miniboone_nested_summary.csv"
MINIBOONE_NESTED_WARNINGS_PATH = DATA_REGISTRY_DIR / "openml_miniboone_nested_warnings.csv"
MINIBOONE_NESTED_ENVIRONMENT_PATH = DATA_REGISTRY_DIR / "openml_miniboone_nested_environment.csv"
MINIBOONE_NESTED_QUALITY_CHECKS_PATH = DATA_REGISTRY_DIR / "openml_miniboone_nested_quality_checks.csv"

required_nested_input_paths = [
    MINIBOONE_NESTED_MODEL_SPACE_PATH,
    MINIBOONE_NESTED_PROTOCOL_LOCK_PATH,
    MINIBOONE_NESTED_EXPECTED_SCHEMA_PATH,
]

missing_nested_input_paths = [path for path in required_nested_input_paths if not path.exists()]
if missing_nested_input_paths:
    raise FileNotFoundError(
        "Не найдены обязательные файлы вложенного протокола: "
        + "; ".join(str(path.relative_to(PROJECT_ROOT)) for path in missing_nested_input_paths)
    )

miniboone_nested_model_space = read_project_csv(MINIBOONE_NESTED_MODEL_SPACE_PATH)
miniboone_nested_protocol_lock = read_project_csv(MINIBOONE_NESTED_PROTOCOL_LOCK_PATH)
miniboone_nested_expected_schema = read_project_csv(MINIBOONE_NESTED_EXPECTED_SCHEMA_PATH)

required_model_space_columns = [
    "protocol_id",
    "candidate_id",
    "model_id",
    "search_method",
    "parameter_name",
    "parameter_values",
    "fixed_value",
    "decision_status",
]

for column in required_model_space_columns:
    if column not in miniboone_nested_model_space.columns:
        raise KeyError(
            f"В {MINIBOONE_NESTED_MODEL_SPACE_PATH.relative_to(PROJECT_ROOT)} "
            f"отсутствует обязательный столбец: {column}"
        )

if not miniboone_nested_model_space["protocol_id"].eq(MINIBOONE_NESTED_PROTOCOL_ID).all():
    raise ValueError("В пространстве параметров найден protocol_id, отличный от miniboone_nested_cv_v01")

if not miniboone_nested_model_space["candidate_id"].eq(MINIBOONE_NESTED_CANDIDATE_ID).all():
    raise ValueError("В пространстве параметров найден candidate_id, отличный от openml_miniboone_41150")

if not miniboone_nested_model_space["decision_status"].eq("locked_before_run").all():
    raise ValueError("Пространство параметров должно быть зафиксировано со статусом locked_before_run")

print("Входы вложенного протокола прочитаны:")
for path in required_nested_input_paths:
    print("OK:", path.relative_to(PROJECT_ROOT))

display(miniboone_nested_model_space)

Входы вложенного протокола прочитаны:
OK: configs\model_spaces\miniboone_hist_gradient_boosting_nested_space.csv
OK: data_registry\openml_miniboone_nested_cv_protocol_lock.csv
OK: data_registry\openml_miniboone_nested_cv_expected_output_schema.csv


,protocol_id,candidate_id,model_id,search_method,parameter_name,parameter_values,fixed_value,decision_status,rationale_ru,source_reference
0,miniboone_nested_cv_v01,openml_miniboone_41150,hist_gradient_boosting,grid,learning_rate,0.05; 0.10,,locked_before_run,Малая сетка вокруг базового значения 0.1 из ми...,REPORT_34 lines 66-79; openml_report35_nested_...
1,miniboone_nested_cv_v01,openml_miniboone_41150,hist_gradient_boosting,grid,max_iter,100; 150,,locked_before_run,Малая сетка вокруг базового значения 100 из ми...,openml_miniboone_model_family_plan.csv line 4;...
2,miniboone_nested_cv_v01,openml_miniboone_41150,hist_gradient_boosting,grid,max_leaf_nodes,31; 63,,locked_before_run,Проверяется базовая сложность дерева и умеренн...,openml_report35_nested_cv_protocol_plan.csv li...
3,miniboone_nested_cv_v01,openml_miniboone_41150,hist_gradient_boosting,grid,l2_regularization,0.0; 0.01,,locked_before_run,Проверяется отсутствие регуляризации и слабая ...,openml_report35_nested_cv_protocol_plan.csv li...
4,miniboone_nested_cv_v01,openml_miniboone_41150,hist_gradient_boosting,fixed,random_state,,20260507,locked_before_run,Сохраняется воспроизводимость относительно пре...,openml_miniboone_model_family_plan.csv line 4
5,miniboone_nested_cv_v01,openml_miniboone_41150,hist_gradient_boosting,fixed,early_stopping,,auto,locked_before_run,Оставляется значение по умолчанию scikit-learn...,scikit-learn HistGradientBoostingClassifier do...


In [ ]:
from mlcra.model_spaces import (
    build_hist_gradient_boosting_search_space,
    make_hgb_parameter_set_id,
)
from mlcra.nested_cv import (
    build_outer_splitter,
    fit_control_estimator_on_outer_block,
    fit_tuned_hist_gradient_boosting_on_outer_block,
    nested_outer_position,
)

In [ ]:
miniboone_nested_parameter_grid, miniboone_nested_fixed_parameters = build_hist_gradient_boosting_search_space(
    miniboone_nested_model_space,
    protocol_id=MINIBOONE_NESTED_PROTOCOL_ID,
    candidate_id=MINIBOONE_NESTED_CANDIDATE_ID,
)

print("Сетка параметров hist_gradient_boosting:")
for parameter_name, values in miniboone_nested_parameter_grid.items():
    print(parameter_name, "=", values)

print("\nФиксированные параметры hist_gradient_boosting:")
for parameter_name, value in miniboone_nested_fixed_parameters.items():
    print(parameter_name, "=", value)

try:
    miniboone_nested_bundle = miniboone_research_bundle
    miniboone_nested_X = miniboone_research_X
    miniboone_nested_y = miniboone_research_y
    miniboone_nested_features = miniboone_research_features
except NameError:
    miniboone_nested_bundle = load_raw_candidate(MINIBOONE_NESTED_CANDIDATE_ID)
    miniboone_nested_X, miniboone_nested_features = build_selected_numeric_frame(
        MINIBOONE_NESTED_CANDIDATE_ID,
        miniboone_nested_bundle["X_raw"],
    )
    miniboone_nested_y = miniboone_nested_bundle["y"]

expected_miniboone_features = [f"ParticleID_{i}" for i in range(50)]
if miniboone_nested_features != expected_miniboone_features:
    raise ValueError(
        "Нарушен замок признаков MiniBooNE: ожидались ParticleID_0 ... ParticleID_49, "
        f"получено: {miniboone_nested_features}"
    )

miniboone_nested_class_counts = pd.Series(miniboone_nested_y).value_counts().sort_index().to_dict()

print("\nMiniBooNE для вложенной проверки:")
print("X:", miniboone_nested_X.shape)
print("Признаки:", len(miniboone_nested_features))
print("Распределение классов:", miniboone_nested_class_counts)
print("Целевые классы:", miniboone_nested_bundle["target_metadata"])

Сетка параметров hist_gradient_boosting:
learning_rate = [0.05, 0.1]
max_iter = [100, 150]
max_leaf_nodes = [31, 63]
l2_regularization = [0.0, 0.01]

Фиксированные параметры hist_gradient_boosting:
random_state = 20260507
early_stopping = auto

MiniBooNE для вложенной проверки:
X: (130064, 50)
Признаки: 50
Распределение классов: {0: 93565, 1: 36499}
Целевые классы: {'target_class_0': 'False', 'target_class_1': 'True', 'positive_class_assumption': 'True'}


In [ ]:
miniboone_nested_outer_score_rows: list[dict[str, Any]] = []
miniboone_nested_selected_param_rows: list[dict[str, Any]] = []
miniboone_nested_warning_rows: list[dict[str, Any]] = []

miniboone_nested_warning_rows.append({
    "severity": "info",
    "candidate_id": MINIBOONE_NESTED_CANDIDATE_ID,
    "protocol_id": MINIBOONE_NESTED_PROTOCOL_ID,
    "outer_split_number": "all",
    "outer_repeat_number": "all",
    "outer_fold_number": "all",
    "model_id": "all",
    "object": "protocol_boundary",
    "warning_ru": (
        "Вложенная проверка использует внешний RepeatedStratifiedKFold 5×2; "
        "выбор параметров hist_gradient_boosting выполняется только во внутреннем StratifiedKFold."
    ),
})

miniboone_nested_outer_cv = build_outer_splitter(
    MINIBOONE_NESTED_N_SPLITS,
    MINIBOONE_NESTED_N_REPEATS,
    MINIBOONE_NESTED_RANDOM_SEED,
)

control_estimators = {
    "dummy_prior": build_estimators()["dummy_prior"],
    "logistic_regression": build_estimators()["logistic_regression"],
}

for outer_split_number, (train_index, test_index) in enumerate(
    miniboone_nested_outer_cv.split(miniboone_nested_X, miniboone_nested_y)
):
    outer_repeat_number, outer_fold_number = nested_outer_position(
        outer_split_number,
        MINIBOONE_NESTED_N_SPLITS,
        MINIBOONE_NESTED_N_REPEATS,
    )

    print("\n" + "=" * 110)
    print(f"MiniBooNE nested CV: repeat={outer_repeat_number}, fold={outer_fold_number}")
    print(
        "train/test:",
        len(train_index),
        len(test_index),
        "positive_share:",
        round(float(np.mean(miniboone_nested_y[train_index])), 6),
        round(float(np.mean(miniboone_nested_y[test_index])), 6),
    )

    tuned_score_row, selected_param_row, captured_warning_rows = fit_tuned_hist_gradient_boosting_on_outer_block(
        candidate_bundle=miniboone_nested_bundle,
        X=miniboone_nested_X,
        y=miniboone_nested_y,
        train_index=train_index,
        test_index=test_index,
        zero_based_outer_split_number=outer_split_number,
        feature_names=miniboone_nested_features,
        parameter_grid=miniboone_nested_parameter_grid,
        fixed_parameters=miniboone_nested_fixed_parameters,
        protocol_id=MINIBOONE_NESTED_PROTOCOL_ID,
        candidate_id=MINIBOONE_NESTED_CANDIDATE_ID,
        outer_n_splits=MINIBOONE_NESTED_N_SPLITS,
        outer_n_repeats=MINIBOONE_NESTED_N_REPEATS,
        inner_n_splits=MINIBOONE_NESTED_INNER_N_SPLITS,
        base_random_state=MINIBOONE_NESTED_RANDOM_SEED,
        feature_policy_id="numeric_particleid_0_49_locked",
        scoring="average_precision",
        refit=True,
        n_jobs=1,
        return_train_score=False,
        error_score="raise",
    )
    miniboone_nested_outer_score_rows.append(tuned_score_row)
    miniboone_nested_selected_param_rows.append(selected_param_row)
    miniboone_nested_warning_rows.extend(captured_warning_rows)

    print(
        "hist_gradient_boosting",
        "average_precision=", f"{tuned_score_row['average_precision']:.6f}",
        "roc_auc=", f"{tuned_score_row['roc_auc']:.6f}",
        "selected_params=", tuned_score_row["selected_params_json"],
    )

    for model_id, estimator_template in control_estimators.items():
        control_score_row, captured_warning_rows = fit_control_estimator_on_outer_block(
            candidate_bundle=miniboone_nested_bundle,
            X=miniboone_nested_X,
            y=miniboone_nested_y,
            train_index=train_index,
            test_index=test_index,
            model_id=model_id,
            estimator_template=estimator_template,
            zero_based_outer_split_number=outer_split_number,
            feature_names=miniboone_nested_features,
            protocol_id=MINIBOONE_NESTED_PROTOCOL_ID,
            candidate_id=MINIBOONE_NESTED_CANDIDATE_ID,
            outer_n_splits=MINIBOONE_NESTED_N_SPLITS,
            outer_n_repeats=MINIBOONE_NESTED_N_REPEATS,
            inner_n_splits=MINIBOONE_NESTED_INNER_N_SPLITS,
            feature_policy_id="numeric_particleid_0_49_locked",
        )
        miniboone_nested_outer_score_rows.append(control_score_row)
        miniboone_nested_warning_rows.extend(captured_warning_rows)

        print(
            model_id,
            "average_precision=", f"{control_score_row['average_precision']:.6f}",
            "roc_auc=", f"{control_score_row['roc_auc']:.6f}",
        )

In [26]:
miniboone_nested_outer_scores_df = pd.DataFrame(miniboone_nested_outer_score_rows)
miniboone_nested_selected_params_df = pd.DataFrame(miniboone_nested_selected_param_rows)
miniboone_nested_warnings_df = pd.DataFrame(miniboone_nested_warning_rows)

expected_outer_blocks = MINIBOONE_NESTED_N_SPLITS * MINIBOONE_NESTED_N_REPEATS
expected_outer_score_rows = expected_outer_blocks * 3
expected_selected_param_rows = expected_outer_blocks

if len(miniboone_nested_outer_scores_df) != expected_outer_score_rows:
    raise ValueError(
        f"Неожиданное число строк внешних оценок: ожидалось {expected_outer_score_rows}, "
        f"получено {len(miniboone_nested_outer_scores_df)}"
    )

if len(miniboone_nested_selected_params_df) != expected_selected_param_rows:
    raise ValueError(
        f"Неожиданное число строк выбранных параметров: ожидалось {expected_selected_param_rows}, "
        f"получено {len(miniboone_nested_selected_params_df)}"
    )

nested_metric_columns = [
    "roc_auc",
    "average_precision",
    "pr_auc",
    "f1",
    "balanced_accuracy",
    "log_loss",
    "brier_score",
]

miniboone_nested_summary_rows: list[dict[str, Any]] = []
for keys, group in miniboone_nested_outer_scores_df.groupby(
    ["protocol_id", "candidate_id", "model_id", "model_role"],
    dropna=False,
):
    protocol_id, candidate_id, model_id, model_role = keys
    row = {
        "protocol_id": protocol_id,
        "candidate_id": candidate_id,
        "model_id": model_id,
        "model_role": model_role,
        "n_external_blocks": int(len(group)),
        "outer_cv_n_splits": MINIBOONE_NESTED_N_SPLITS,
        "outer_cv_n_repeats": MINIBOONE_NESTED_N_REPEATS,
        "inner_cv_n_splits": MINIBOONE_NESTED_INNER_N_SPLITS if model_role == "tuned_candidate" else "",
        "random_state": MINIBOONE_NESTED_RANDOM_SEED,
        "row_status": "nested_research_draft",
        "interpretation_allowed": "limited_nested_protocol_review_only",
    }
    for metric in nested_metric_columns:
        row[f"{metric}_mean"] = float(group[metric].mean())
        row[f"{metric}_std"] = float(group[metric].std(ddof=0))
        row[f"{metric}_min"] = float(group[metric].min())
        row[f"{metric}_max"] = float(group[metric].max())
    miniboone_nested_summary_rows.append(row)

miniboone_nested_summary_df = pd.DataFrame(miniboone_nested_summary_rows).sort_values(
    ["candidate_id", "model_id"],
    kind="mergesort",
)

print("Строки внешних оценок:", len(miniboone_nested_outer_scores_df))
print("Строки выбранных параметров:", len(miniboone_nested_selected_params_df))
print("\nСводка вложенной проверки MiniBooNE:")
display(miniboone_nested_summary_df)

print("\nВыбранные параметры hist_gradient_boosting:")
display(miniboone_nested_selected_params_df)

print("\nПредупреждения вложенной проверки:")
display(miniboone_nested_warnings_df)

Строки внешних оценок: 30
Строки выбранных параметров: 10

Сводка вложенной проверки MiniBooNE:


,protocol_id,candidate_id,model_id,model_role,n_external_blocks,outer_cv_n_splits,outer_cv_n_repeats,inner_cv_n_splits,random_state,row_status,interpretation_allowed,roc_auc_mean,roc_auc_std,roc_auc_min,roc_auc_max,average_precision_mean,average_precision_std,average_precision_min,average_precision_max,pr_auc_mean,pr_auc_std,pr_auc_min,pr_auc_max,f1_mean,f1_std,f1_min,f1_max,balanced_accuracy_mean,balanced_accuracy_std,balanced_accuracy_min,balanced_accuracy_max,log_loss_mean,log_loss_std,log_loss_min,log_loss_max,brier_score_mean,brier_score_std,brier_score_min,brier_score_max
0,miniboone_nested_cv_v01,openml_miniboone_41150,dummy_prior,control,10,5,2,,20260507,nested_research_draft,limited_nested_protocol_review_only,0.500000,0.000000,0.500000,0.500000,0.280623,0.000011,0.280601,0.280629,0.280623,0.000011,0.280601,0.280629,0.000000,0.000000,0.000000,0.000000,0.500000,0.000000,0.500000,0.500000,0.593541,0.000010,0.593520,0.593546,0.201874,0.000005,0.201864,0.201876
1,miniboone_nested_cv_v01,openml_miniboone_41150,hist_gradient_boosting,tuned_candidate,10,5,2,3,20260507,nested_research_draft,limited_nested_protocol_review_only,0.985045,0.000663,0.983971,0.986157,0.959310,0.002259,0.956287,0.962441,0.959310,0.002259,0.956287,0.962441,0.902052,0.002442,0.897370,0.905913,0.933761,0.001298,0.931099,0.935359,0.139541,0.002905,0.134483,0.144375,0.041102,0.000912,0.039419,0.042552
2,miniboone_nested_cv_v01,openml_miniboone_41150,logistic_regression,control,10,5,2,,20260507,nested_research_draft,limited_nested_protocol_review_only,0.939278,0.001284,0.936721,0.941325,0.867408,0.002849,0.862086,0.872183,0.867408,0.002849,0.862086,0.872183,0.776990,0.002756,0.772805,0.782049,0.834122,0.001872,0.830738,0.837224,0.285299,0.002487,0.281366,0.290135,0.085489,0.000767,0.084325,0.087380



Выбранные параметры hist_gradient_boosting:


,protocol_id,candidate_id,outer_split_number,outer_repeat_number,outer_fold_number,model_id,selected_parameter_set_id,selected_params_json,inner_best_average_precision,inner_cv_n_splits,inner_cv_random_state,inner_candidate_count,row_status
0,miniboone_nested_cv_v01,openml_miniboone_41150,1,1,1,hist_gradient_boosting,hgb_b5168b128e78,"{""l2_regularization"": 0.0, ""learning_rate"": 0....",0.959059,3,20260507,16,nested_research_draft
1,miniboone_nested_cv_v01,openml_miniboone_41150,2,1,2,hist_gradient_boosting,hgb_b5168b128e78,"{""l2_regularization"": 0.0, ""learning_rate"": 0....",0.957707,3,20260508,16,nested_research_draft
2,miniboone_nested_cv_v01,openml_miniboone_41150,3,1,3,hist_gradient_boosting,hgb_b5168b128e78,"{""l2_regularization"": 0.0, ""learning_rate"": 0....",0.956787,3,20260509,16,nested_research_draft
3,miniboone_nested_cv_v01,openml_miniboone_41150,4,1,4,hist_gradient_boosting,hgb_8316cb7622a5,"{""l2_regularization"": 0.01, ""learning_rate"": 0...",0.958203,3,20260510,16,nested_research_draft
4,miniboone_nested_cv_v01,openml_miniboone_41150,5,1,5,hist_gradient_boosting,hgb_b5168b128e78,"{""l2_regularization"": 0.0, ""learning_rate"": 0....",0.957837,3,20260511,16,nested_research_draft
5,miniboone_nested_cv_v01,openml_miniboone_41150,6,2,1,hist_gradient_boosting,hgb_b5168b128e78,"{""l2_regularization"": 0.0, ""learning_rate"": 0....",0.958599,3,20260512,16,nested_research_draft
6,miniboone_nested_cv_v01,openml_miniboone_41150,7,2,2,hist_gradient_boosting,hgb_b5168b128e78,"{""l2_regularization"": 0.0, ""learning_rate"": 0....",0.957174,3,20260513,16,nested_research_draft
7,miniboone_nested_cv_v01,openml_miniboone_41150,8,2,3,hist_gradient_boosting,hgb_8316cb7622a5,"{""l2_regularization"": 0.01, ""learning_rate"": 0...",0.957980,3,20260514,16,nested_research_draft
8,miniboone_nested_cv_v01,openml_miniboone_41150,9,2,4,hist_gradient_boosting,hgb_b5168b128e78,"{""l2_regularization"": 0.0, ""learning_rate"": 0....",0.958102,3,20260515,16,nested_research_draft
9,miniboone_nested_cv_v01,openml_miniboone_41150,10,2,5,hist_gradient_boosting,hgb_8316cb7622a5,"{""l2_regularization"": 0.01, ""learning_rate"": 0...",0.957374,3,20260516,16,nested_research_draft



Предупреждения вложенной проверки:


,severity,candidate_id,protocol_id,outer_split_number,outer_repeat_number,outer_fold_number,model_id,object,warning_ru
0,info,openml_miniboone_41150,miniboone_nested_cv_v01,all,all,all,all,protocol_boundary,Вложенная проверка использует внешний Repeated...


In [27]:
def make_quality_check_row(check_id: str, condition: bool, details_ru: str) -> dict[str, Any]:
    return {
        "protocol_id": MINIBOONE_NESTED_PROTOCOL_ID,
        "check_id": check_id,
        "check_result": "pass" if condition else "fail",
        "details_ru": details_ru,
    }


def required_columns_for_output(file_path: str) -> list[str]:
    rows = miniboone_nested_expected_schema[
        miniboone_nested_expected_schema["file_path"].eq(file_path)
        & miniboone_nested_expected_schema["required"].eq("yes")
    ].copy()
    return rows["column_name"].tolist()


def check_required_columns(file_path: str, frame: pd.DataFrame) -> bool:
    missing = [column for column in required_columns_for_output(file_path) if column not in frame.columns]
    return len(missing) == 0


miniboone_nested_environment_rows = []
for package_name in ["python", "platform", "openml", "sklearn", "pandas", "numpy"]:
    if package_name == "python":
        version = sys.version.replace("\n", " ")
    elif package_name == "platform":
        version = platform.platform()
    else:
        module_name = "sklearn" if package_name == "sklearn" else package_name
        module = importlib.import_module(module_name)
        version = getattr(module, "__version__", "unknown")
    miniboone_nested_environment_rows.append({
        "name": package_name,
        "version": version,
        "protocol_id": MINIBOONE_NESTED_PROTOCOL_ID,
    })

miniboone_nested_environment_df = pd.DataFrame(miniboone_nested_environment_rows)

valid_hgb_grid_values = {
    parameter_name: set(values)
    for parameter_name, values in miniboone_nested_parameter_grid.items()
}

selected_params_within_locked_grid = True
for raw_params in miniboone_nested_selected_params_df["selected_params_json"]:
    params = json.loads(raw_params)
    for parameter_name, parameter_value in params.items():
        if parameter_value not in valid_hgb_grid_values.get(parameter_name, set()):
            selected_params_within_locked_grid = False

miniboone_nested_quality_rows = [
    make_quality_check_row(
        "input_model_space_exists",
        MINIBOONE_NESTED_MODEL_SPACE_PATH.exists(),
        "Файл пространства параметров существует.",
    ),
    make_quality_check_row(
        "input_protocol_lock_exists",
        MINIBOONE_NESTED_PROTOCOL_LOCK_PATH.exists(),
        "Файл фиксации протокола существует.",
    ),
    make_quality_check_row(
        "outer_scores_row_count",
        len(miniboone_nested_outer_scores_df) == expected_outer_score_rows,
        f"Ожидалось {expected_outer_score_rows} строк внешних оценок.",
    ),
    make_quality_check_row(
        "selected_params_row_count",
        len(miniboone_nested_selected_params_df) == expected_selected_param_rows,
        f"Ожидалось {expected_selected_param_rows} строк выбранных параметров.",
    ),
    make_quality_check_row(
        "summary_row_count",
        len(miniboone_nested_summary_df) == 3,
        "Ожидалось 3 строки сводки: hist_gradient_boosting, logistic_regression, dummy_prior.",
    ),
    make_quality_check_row(
        "feature_count_locked",
        miniboone_nested_outer_scores_df["feature_count"].eq(50).all(),
        "Во всех строках должно быть feature_count=50.",
    ),
    make_quality_check_row(
        "row_status_locked",
        miniboone_nested_outer_scores_df["row_status"].eq("nested_research_draft").all(),
        "Все строки внешних оценок должны иметь row_status=nested_research_draft.",
    ),
    make_quality_check_row(
        "hist_gradient_boosting_selected_params_within_locked_grid",
        selected_params_within_locked_grid,
        "Все выбранные параметры hist_gradient_boosting должны входить в заранее зафиксированную сетку.",
    ),
    make_quality_check_row(
        "outer_scores_required_columns",
        check_required_columns(
            "data_registry/openml_miniboone_nested_outer_scores.csv",
            miniboone_nested_outer_scores_df,
        ),
        "Файл внешних оценок содержит обязательные столбцы из ожидаемой схемы.",
    ),
    make_quality_check_row(
        "selected_params_required_columns",
        check_required_columns(
            "data_registry/openml_miniboone_nested_selected_params.csv",
            miniboone_nested_selected_params_df,
        ),
        "Файл выбранных параметров содержит обязательные столбцы из ожидаемой схемы.",
    ),
    make_quality_check_row(
        "summary_required_columns",
        check_required_columns(
            "data_registry/openml_miniboone_nested_summary.csv",
            miniboone_nested_summary_df,
        ),
        "Файл сводки содержит обязательные столбцы из ожидаемой схемы.",
    ),
    make_quality_check_row(
        "warnings_required_columns",
        check_required_columns(
            "data_registry/openml_miniboone_nested_warnings.csv",
            miniboone_nested_warnings_df,
        ),
        "Файл предупреждений содержит обязательные столбцы из ожидаемой схемы.",
    ),
    make_quality_check_row(
        "environment_required_columns",
        check_required_columns(
            "data_registry/openml_miniboone_nested_environment.csv",
            miniboone_nested_environment_df,
        ),
        "Файл среды содержит обязательные столбцы из ожидаемой схемы.",
    ),
]

miniboone_nested_quality_checks_df = pd.DataFrame(miniboone_nested_quality_rows)

if miniboone_nested_quality_checks_df["check_result"].eq("fail").any():
    display(miniboone_nested_quality_checks_df)
    raise ValueError("Одна или несколько проверок полноты вложенного протокола завершились fail")

miniboone_nested_outer_scores_df.to_csv(
    MINIBOONE_NESTED_OUTER_SCORES_PATH,
    index=False,
    encoding="utf-8-sig",
)
miniboone_nested_selected_params_df.to_csv(
    MINIBOONE_NESTED_SELECTED_PARAMS_PATH,
    index=False,
    encoding="utf-8-sig",
)
miniboone_nested_summary_df.to_csv(
    MINIBOONE_NESTED_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig",
)
miniboone_nested_warnings_df.to_csv(
    MINIBOONE_NESTED_WARNINGS_PATH,
    index=False,
    encoding="utf-8-sig",
)
miniboone_nested_environment_df.to_csv(
    MINIBOONE_NESTED_ENVIRONMENT_PATH,
    index=False,
    encoding="utf-8-sig",
)
miniboone_nested_quality_checks_df.to_csv(
    MINIBOONE_NESTED_QUALITY_CHECKS_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Сохранены файлы вложенной проверки MiniBooNE:")
print(MINIBOONE_NESTED_OUTER_SCORES_PATH.relative_to(PROJECT_ROOT))
print(MINIBOONE_NESTED_SELECTED_PARAMS_PATH.relative_to(PROJECT_ROOT))
print(MINIBOONE_NESTED_SUMMARY_PATH.relative_to(PROJECT_ROOT))
print(MINIBOONE_NESTED_WARNINGS_PATH.relative_to(PROJECT_ROOT))
print(MINIBOONE_NESTED_ENVIRONMENT_PATH.relative_to(PROJECT_ROOT))
print(MINIBOONE_NESTED_QUALITY_CHECKS_PATH.relative_to(PROJECT_ROOT))

print("\nПроверки полноты:")
display(miniboone_nested_quality_checks_df)

print("\nКонтроль границ:")
print("1. Внешние блоки не использовались для выбора параметров.")
print("2. Параметры hist_gradient_boosting выбирались только во внутреннем контуре.")
print("3. logistic_regression и dummy_prior остались контрольными моделями без настройки.")
print("4. dataset_candidate_registry.csv не изменялся этой тетрадью.")
print("5. Статус результатов: nested_research_draft.")

Сохранены файлы вложенной проверки MiniBooNE:
data_registry\openml_miniboone_nested_outer_scores.csv
data_registry\openml_miniboone_nested_selected_params.csv
data_registry\openml_miniboone_nested_summary.csv
data_registry\openml_miniboone_nested_warnings.csv
data_registry\openml_miniboone_nested_environment.csv
data_registry\openml_miniboone_nested_quality_checks.csv

Проверки полноты:


,protocol_id,check_id,check_result,details_ru
0,miniboone_nested_cv_v01,input_model_space_exists,pass,Файл пространства параметров существует.
1,miniboone_nested_cv_v01,input_protocol_lock_exists,pass,Файл фиксации протокола существует.
2,miniboone_nested_cv_v01,outer_scores_row_count,pass,Ожидалось 30 строк внешних оценок.
3,miniboone_nested_cv_v01,selected_params_row_count,pass,Ожидалось 10 строк выбранных параметров.
4,miniboone_nested_cv_v01,summary_row_count,pass,Ожидалось 3 строки сводки: hist_gradient_boost...
5,miniboone_nested_cv_v01,feature_count_locked,pass,Во всех строках должно быть feature_count=50.
6,miniboone_nested_cv_v01,row_status_locked,pass,Все строки внешних оценок должны иметь row_sta...
7,miniboone_nested_cv_v01,hist_gradient_boosting_selected_params_within_...,pass,Все выбранные параметры hist_gradient_boosting...
8,miniboone_nested_cv_v01,outer_scores_required_columns,pass,Файл внешних оценок содержит обязательные стол...
9,miniboone_nested_cv_v01,selected_params_required_columns,pass,Файл выбранных параметров содержит обязательны...



Контроль границ:
1. Внешние блоки не использовались для выбора параметров.
2. Параметры hist_gradient_boosting выбирались только во внутреннем контуре.
3. logistic_regression и dummy_prior остались контрольными моделями без настройки.
4. dataset_candidate_registry.csv не изменялся этой тетрадью.
5. Статус результатов: nested_research_draft.


## 8 — Этап 5: текущий срез эффекта без нового обучения

Цель блока: сформировать машинно-читаемый срез текущего эффекта `hist_gradient_boosting` относительно контрольных моделей по уже выполненной вложенной проверке MiniBooNE v01.

Границы блока:

1. новые модели не обучаются;
2. пространство параметров не изменяется;
3. файл `dataset_candidate_registry.csv` не изменяется;
4. результат используется как вход этапа 5, а не как итоговый вердикт.

In [ ]:
from mlcra.verdicts import build_current_effect_readout

MINIBOONE_STAGE05_CLAIM_ID = "miniboone_hgb_vs_logreg_average_precision_v01"
MINIBOONE_STAGE05_CURRENT_READOUT_ID = "ST05_01_current_nested_readout"

MINIBOONE_STAGE05_CLAIM_PATH = (
    PROJECT_ROOT
    / "configs"
    / "claims"
    / "miniboone_hgb_vs_logreg_claim_v01.csv"
)
MINIBOONE_STAGE05_STRESS_TEST_PLAN_PATH = (
    PROJECT_ROOT
    / "configs"
    / "stress_tests"
    / "miniboone_stress_test_plan_v01.csv"
)
MINIBOONE_STAGE05_VERDICT_POLICY_PATH = (
    PROJECT_ROOT
    / "configs"
    / "verdict_policies"
    / "miniboone_verdict_policy_v01.csv"
)

MINIBOONE_STAGE05_SOURCE_OUTER_SCORES_PATH = DATA_REGISTRY_DIR / "openml_miniboone_nested_outer_scores.csv"
MINIBOONE_STAGE05_CURRENT_EFFECT_READOUT_PATH = (
    DATA_REGISTRY_DIR
    / "openml_miniboone_stage05_current_effect_readout.csv"
)

required_stage05_input_paths = [
    MINIBOONE_STAGE05_CLAIM_PATH,
    MINIBOONE_STAGE05_STRESS_TEST_PLAN_PATH,
    MINIBOONE_STAGE05_VERDICT_POLICY_PATH,
    MINIBOONE_STAGE05_SOURCE_OUTER_SCORES_PATH,
]

missing_stage05_input_paths = [
    path for path in required_stage05_input_paths
    if not path.exists()
]

if missing_stage05_input_paths:
    raise FileNotFoundError(
        "Не найдены обязательные входные файлы этапа 5: "
        + "; ".join(str(path.relative_to(PROJECT_ROOT)) for path in missing_stage05_input_paths)
    )

miniboone_stage05_claim = read_project_csv(MINIBOONE_STAGE05_CLAIM_PATH)
miniboone_stage05_stress_plan = read_project_csv(MINIBOONE_STAGE05_STRESS_TEST_PLAN_PATH)
miniboone_stage05_verdict_policy = read_project_csv(MINIBOONE_STAGE05_VERDICT_POLICY_PATH)
miniboone_stage05_outer_scores = read_project_csv(MINIBOONE_STAGE05_SOURCE_OUTER_SCORES_PATH)

claim_rows = miniboone_stage05_claim[
    miniboone_stage05_claim["claim_id"].eq(MINIBOONE_STAGE05_CLAIM_ID)
].copy()

if len(claim_rows) != 1:
    raise ValueError(
        f"Ожидалась ровно одна строка claim_id={MINIBOONE_STAGE05_CLAIM_ID}, "
        f"получено: {len(claim_rows)}"
    )

stress_rows = miniboone_stage05_stress_plan[
    miniboone_stage05_stress_plan["stress_test_id"].eq(MINIBOONE_STAGE05_CURRENT_READOUT_ID)
].copy()

if len(stress_rows) != 1:
    raise ValueError(
        f"Ожидалась ровно одна строка stress_test_id={MINIBOONE_STAGE05_CURRENT_READOUT_ID}, "
        f"получено: {len(stress_rows)}"
    )

if stress_rows["requires_new_model_run"].iloc[0] != "no":
    raise ValueError("Текущий блок этапа 5 должен иметь requires_new_model_run=no")

required_outer_score_columns = [
    "protocol_id",
    "candidate_id",
    "outer_split_number",
    "outer_repeat_number",
    "outer_fold_number",
    "model_id",
    "average_precision",
    "roc_auc",
    "f1",
    "balanced_accuracy",
    "log_loss",
    "brier_score",
    "row_status",
]

for column in required_outer_score_columns:
    if column not in miniboone_stage05_outer_scores.columns:
        raise KeyError(
            f"В {MINIBOONE_STAGE05_SOURCE_OUTER_SCORES_PATH.relative_to(PROJECT_ROOT)} "
            f"отсутствует обязательный столбец: {column}"
        )

if not miniboone_stage05_outer_scores["row_status"].eq("nested_research_draft").all():
    raise ValueError("Все строки исходной вложенной проверки должны иметь row_status=nested_research_draft")

expected_stage05_models = {
    "hist_gradient_boosting",
    "logistic_regression",
    "dummy_prior",
}

actual_stage05_models = set(miniboone_stage05_outer_scores["model_id"].unique())

if actual_stage05_models != expected_stage05_models:
    raise ValueError(
        "Список моделей в исходных внешних оценках не совпадает с ожидаемым. "
        f"Ожидалось: {sorted(expected_stage05_models)}, получено: {sorted(actual_stage05_models)}"
    )

stage05_block_counts = miniboone_stage05_outer_scores.groupby(
    ["outer_split_number", "outer_repeat_number", "outer_fold_number"],
    dropna=False,
)["model_id"].nunique()

if len(stage05_block_counts) != 10:
    raise ValueError(f"Ожидалось 10 внешних блоков, получено: {len(stage05_block_counts)}")

if not stage05_block_counts.eq(3).all():
    raise ValueError("В каждом внешнем блоке должны присутствовать 3 модели")

print("Входы ST05_01 прочитаны:")
for path in required_stage05_input_paths:
    print("OK:", path.relative_to(PROJECT_ROOT))

print("\nПроверка структуры исходных внешних оценок:")
print("Строк:", len(miniboone_stage05_outer_scores))
print("Внешних блоков:", len(stage05_block_counts))
print("Модели:", sorted(actual_stage05_models))

Входы ST05_01 прочитаны:
OK: configs\claims\miniboone_hgb_vs_logreg_claim_v01.csv
OK: configs\stress_tests\miniboone_stress_test_plan_v01.csv
OK: configs\verdict_policies\miniboone_verdict_policy_v01.csv
OK: data_registry\openml_miniboone_nested_outer_scores.csv

Проверка структуры исходных внешних оценок:
Строк: 30
Внешних блоков: 10
Модели: ['dummy_prior', 'hist_gradient_boosting', 'logistic_regression']


In [ ]:
miniboone_stage05_current_effect_df = build_current_effect_readout(
    miniboone_stage05_outer_scores
)

expected_stage05_paired_rows = 2 * 6 * 10
expected_stage05_summary_rows = 2 * 6
expected_stage05_total_rows = expected_stage05_paired_rows + expected_stage05_summary_rows

if len(miniboone_stage05_current_effect_df) != expected_stage05_total_rows:
    raise ValueError(
        f"Неожиданное число строк текущего среза эффекта: "
        f"ожидалось {expected_stage05_total_rows}, "
        f"получено {len(miniboone_stage05_current_effect_df)}"
    )

primary_average_precision_summary = miniboone_stage05_current_effect_df[
    miniboone_stage05_current_effect_df["row_type"].eq("metric_summary")
    & miniboone_stage05_current_effect_df["comparison_model_id"].eq("logistic_regression")
    & miniboone_stage05_current_effect_df["metric_name"].eq("average_precision")
].copy()

if len(primary_average_precision_summary) != 1:
    raise ValueError("Не удалось однозначно получить сводку HGB против logistic_regression по average_precision")

if int(primary_average_precision_summary["candidate_positive_blocks"].iloc[0]) != 10:
    raise ValueError(
        "По текущему вложенному результату HGB не превосходит logistic_regression "
        "по average_precision во всех 10 внешних блоках"
    )

miniboone_stage05_current_effect_df.to_csv(
    MINIBOONE_STAGE05_CURRENT_EFFECT_READOUT_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Сохранен текущий срез эффекта этапа 5:")
print(MINIBOONE_STAGE05_CURRENT_EFFECT_READOUT_PATH.relative_to(PROJECT_ROOT))

print("\nРазмер результата:")
print("Строк парных внешних блоков:", expected_stage05_paired_rows)
print("Строк сводки:", expected_stage05_summary_rows)
print("Всего строк:", len(miniboone_stage05_current_effect_df))

print("\nСводка по метрикам:")
display(
    miniboone_stage05_current_effect_df[
        miniboone_stage05_current_effect_df["row_type"].eq("metric_summary")
    ][
        [
            "comparison_model_id",
            "metric_name",
            "metric_direction",
            "n_blocks",
            "advantage_mean",
            "advantage_min",
            "advantage_max",
            "candidate_positive_blocks",
            "candidate_negative_blocks",
        ]
    ].sort_values(
        ["comparison_model_id", "metric_name"],
        kind="mergesort",
    )
)

print("\nКонтроль границ:")
print("1. Новые модели не обучались.")
print("2. Пространство параметров не изменялось.")
print("3. Использованы только уже сохраненные внешние оценки вложенной проверки.")
print("4. dataset_candidate_registry.csv не изменялся.")
print("5. Результат является входом этапа 5, а не итоговым вердиктом.")

Сохранен текущий срез эффекта этапа 5:
data_registry\openml_miniboone_stage05_current_effect_readout.csv

Размер результата:
Строк парных внешних блоков: 120
Строк сводки: 12
Всего строк: 132

Сводка по метрикам:


,comparison_model_id,metric_name,metric_direction,n_blocks,advantage_mean,advantage_min,advantage_max,candidate_positive_blocks,candidate_negative_blocks
120,dummy_prior,average_precision,higher_is_better,10,0.678686,0.675658,0.681812,10,0
121,dummy_prior,balanced_accuracy,higher_is_better,10,0.433761,0.431099,0.435359,10,0
122,dummy_prior,brier_score,lower_is_better,10,0.160772,0.159324,0.162458,10,0
123,dummy_prior,f1,higher_is_better,10,0.902052,0.89737,0.905913,10,0
124,dummy_prior,log_loss,lower_is_better,10,0.454,0.449171,0.459063,10,0
125,dummy_prior,roc_auc,higher_is_better,10,0.485045,0.483971,0.486157,10,0
126,logistic_regression,average_precision,higher_is_better,10,0.091901,0.090258,0.094269,10,0
127,logistic_regression,balanced_accuracy,higher_is_better,10,0.099639,0.097101,0.104622,10,0
128,logistic_regression,brier_score,lower_is_better,10,0.044387,0.043356,0.046326,10,0
129,logistic_regression,f1,higher_is_better,10,0.125062,0.122401,0.133108,10,0



Контроль границ:
1. Новые модели не обучались.
2. Пространство параметров не изменялось.
3. Использованы только уже сохраненные внешние оценки вложенной проверки.
4. dataset_candidate_registry.csv не изменялся.
5. Результат является входом этапа 5, а не итоговым вердиктом.


## 9 — Этап 5: устойчивость к зернам случайности

Цель блока: выполнить заранее зафиксированный стресс-тест `ST05_02_seed_stability_grid`.

Проверяется не новая гипотеза и не новое пространство параметров, а устойчивость уже зафиксированного утверждения к пяти зернам случайности внешнего разбиения.

Границы блока:

1. используется только зафиксированная сетка параметров `hist_gradient_boosting`;
2. внешний протокол остается `RepeatedStratifiedKFold 5×2`;
3. внутренний протокол остается `StratifiedKFold` с 3 блоками;
4. сравнение ведется попарно внутри каждого внешнего блока;
5. итоговый вердикт этапа 5 этим блоком не выносится.

In [ ]:
from sklearn.model_selection import ParameterGrid

from mlcra.io import read_csv_checked, write_csv_checked
from mlcra.model_spaces import build_hist_gradient_boosting_search_space
from mlcra.stress_tests import (
    STAGE05_SEED_CANDIDATE_ID,
    build_seed_stability_contract,
    build_seed_stability_control_estimators,
    run_seed_stability_grid,
)

MINIBOONE_STAGE05_CLAIM_PATH = (
    PROJECT_ROOT / "configs" / "claims" / "miniboone_hgb_vs_logreg_claim_v01.csv"
)
MINIBOONE_STAGE05_STRESS_TEST_PLAN_PATH = (
    PROJECT_ROOT / "configs" / "stress_tests" / "miniboone_stress_test_plan_v01.csv"
)
MINIBOONE_STAGE05_MODEL_SPACE_PATH = (
    PROJECT_ROOT / "configs" / "model_spaces" / "miniboone_hist_gradient_boosting_nested_space.csv"
)
MINIBOONE_STAGE05_SEED_OUTER_SCORES_PATH = (
    DATA_REGISTRY_DIR / "openml_miniboone_stage05_seed_stability_outer_scores.csv"
)
MINIBOONE_STAGE05_SEED_SELECTED_PARAMS_PATH = (
    DATA_REGISTRY_DIR / "openml_miniboone_stage05_seed_stability_selected_params.csv"
)
MINIBOONE_STAGE05_SEED_SUMMARY_PATH = (
    DATA_REGISTRY_DIR / "openml_miniboone_stage05_seed_stability_summary.csv"
)

miniboone_stage05_seed_claim = read_csv_checked(MINIBOONE_STAGE05_CLAIM_PATH)
miniboone_stage05_seed_plan = read_csv_checked(MINIBOONE_STAGE05_STRESS_TEST_PLAN_PATH)
miniboone_stage05_seed_model_space = read_csv_checked(MINIBOONE_STAGE05_MODEL_SPACE_PATH)
stage05_seed_contract = build_seed_stability_contract(
    miniboone_stage05_seed_plan,
    miniboone_stage05_seed_claim,
)
print(stage05_seed_contract)

In [ ]:
stage05_seed_parameter_grid, stage05_seed_fixed_parameters = (
    build_hist_gradient_boosting_search_space(
        miniboone_stage05_seed_model_space,
        protocol_id=stage05_seed_contract.nested_protocol_id,
        candidate_id=stage05_seed_contract.candidate_id,
    )
)
stage05_seed_control_estimators = build_seed_stability_control_estimators()

print("HGB parameter candidates:", len(list(ParameterGrid(stage05_seed_parameter_grid))))
print("HGB fixed parameters:", stage05_seed_fixed_parameters)

In [ ]:
try:
    miniboone_stage05_seed_bundle = miniboone_nested_bundle
    miniboone_stage05_seed_X = miniboone_nested_X
    miniboone_stage05_seed_y = miniboone_nested_y
    miniboone_stage05_seed_features = miniboone_nested_features
except NameError:
    try:
        miniboone_stage05_seed_bundle = miniboone_research_bundle
        miniboone_stage05_seed_X = miniboone_research_X
        miniboone_stage05_seed_y = miniboone_research_y
        miniboone_stage05_seed_features = miniboone_research_features
    except NameError:
        miniboone_stage05_seed_bundle = load_raw_candidate(STAGE05_SEED_CANDIDATE_ID)
        miniboone_stage05_seed_X, miniboone_stage05_seed_features = build_selected_numeric_frame(
            STAGE05_SEED_CANDIDATE_ID,
            miniboone_stage05_seed_bundle["X_raw"],
        )
        miniboone_stage05_seed_y = miniboone_stage05_seed_bundle["y"]

stage05_seed_execution_bundle = run_seed_stability_grid(
    contract=stage05_seed_contract,
    candidate_bundle=miniboone_stage05_seed_bundle,
    X=miniboone_stage05_seed_X,
    y=miniboone_stage05_seed_y,
    feature_names=miniboone_stage05_seed_features,
    parameter_grid=stage05_seed_parameter_grid,
    fixed_parameters=stage05_seed_fixed_parameters,
    control_estimators=stage05_seed_control_estimators,
)

In [ ]:
stage05_seed_outer_scores_df = stage05_seed_execution_bundle.outer_scores
stage05_seed_selected_params_df = stage05_seed_execution_bundle.selected_params
stage05_seed_summary_df = stage05_seed_execution_bundle.summary

write_csv_checked(stage05_seed_outer_scores_df, MINIBOONE_STAGE05_SEED_OUTER_SCORES_PATH)
write_csv_checked(stage05_seed_selected_params_df, MINIBOONE_STAGE05_SEED_SELECTED_PARAMS_PATH)
write_csv_checked(stage05_seed_summary_df, MINIBOONE_STAGE05_SEED_SUMMARY_PATH)

print(MINIBOONE_STAGE05_SEED_OUTER_SCORES_PATH.relative_to(PROJECT_ROOT))
print(MINIBOONE_STAGE05_SEED_SELECTED_PARAMS_PATH.relative_to(PROJECT_ROOT))
print(MINIBOONE_STAGE05_SEED_SUMMARY_PATH.relative_to(PROJECT_ROOT))
print("outer scores:", len(stage05_seed_outer_scores_df))
print("selected parameters:", len(stage05_seed_selected_params_df))
print("summary rows:", len(stage05_seed_summary_df))
display(
    stage05_seed_summary_df[
        stage05_seed_summary_df["comparison_model_id"].eq("logistic_regression")
        & stage05_seed_summary_df["metric_name"].eq("average_precision")
    ]
)


## 10 — Этап 5: чувствительность к внешнему протоколу 10×1

Цель блока: выполнить заранее зафиксированный стресс-тест `ST05_03a_split_protocol_10x1`.

Проверяется не новое утверждение и не новое пространство параметров, а чувствительность уже зафиксированного утверждения `hist_gradient_boosting > logistic_regression` к альтернативному внешнему протоколу разбиения.

Границы блока:

1. используется только зафиксированная сетка параметров `hist_gradient_boosting`;
2. внешний протокол изменяется на `RepeatedStratifiedKFold 10×1`;
3. внутренний протокол остается `StratifiedKFold` с 3 блоками;
4. сравнение ведется попарно внутри каждого внешнего блока;
5. результаты `10×1` не объединяются с результатами `5×2` без отдельного evidence record;
6. итоговый вердикт этапа 5 этим блоком не выносится.

In [ ]:
from sklearn.model_selection import ParameterGrid

from mlcra.io import read_csv_checked, write_csv_checked
from mlcra.model_spaces import build_hist_gradient_boosting_search_space
from mlcra.stress_tests import (
    STAGE05_SPLIT_10X1_CANDIDATE_ID,
    build_split_10x1_contract,
    build_split_10x1_control_estimators,
    run_split_10x1_grid,
)

MINIBOONE_STAGE05_SPLIT_10X1_CLAIM_PATH = (
    PROJECT_ROOT / "configs" / "claims" / "miniboone_hgb_vs_logreg_claim_v01.csv"
)
MINIBOONE_STAGE05_SPLIT_10X1_PLAN_PATH = (
    PROJECT_ROOT / "configs" / "stress_tests" / "miniboone_stress_test_plan_v01.csv"
)
MINIBOONE_STAGE05_SPLIT_10X1_MODEL_SPACE_PATH = (
    PROJECT_ROOT / "configs" / "model_spaces" / "miniboone_hist_gradient_boosting_nested_space.csv"
)
MINIBOONE_STAGE05_SPLIT_10X1_OUTER_SCORES_PATH = (
    DATA_REGISTRY_DIR / "openml_miniboone_stage05_split_10x1_outer_scores.csv"
)
MINIBOONE_STAGE05_SPLIT_10X1_SUMMARY_PATH = (
    DATA_REGISTRY_DIR / "openml_miniboone_stage05_split_10x1_summary.csv"
)

miniboone_stage05_split_10x1_claim = read_csv_checked(
    MINIBOONE_STAGE05_SPLIT_10X1_CLAIM_PATH
)
miniboone_stage05_split_10x1_plan = read_csv_checked(
    MINIBOONE_STAGE05_SPLIT_10X1_PLAN_PATH
)
miniboone_stage05_split_10x1_model_space = read_csv_checked(
    MINIBOONE_STAGE05_SPLIT_10X1_MODEL_SPACE_PATH
)
stage05_split_10x1_contract = build_split_10x1_contract(
    miniboone_stage05_split_10x1_plan,
    miniboone_stage05_split_10x1_claim,
)
print(stage05_split_10x1_contract)


In [ ]:
stage05_split_10x1_parameter_grid, stage05_split_10x1_fixed_parameters = (
    build_hist_gradient_boosting_search_space(
        miniboone_stage05_split_10x1_model_space,
        protocol_id=stage05_split_10x1_contract.nested_protocol_id,
        candidate_id=stage05_split_10x1_contract.candidate_id,
    )
)
stage05_split_10x1_control_estimators = build_split_10x1_control_estimators()

try:
    miniboone_stage05_split_10x1_bundle = miniboone_stage05_seed_bundle
    miniboone_stage05_split_10x1_X = miniboone_stage05_seed_X
    miniboone_stage05_split_10x1_y = miniboone_stage05_seed_y
    miniboone_stage05_split_10x1_features = miniboone_stage05_seed_features
except NameError:
    try:
        miniboone_stage05_split_10x1_bundle = miniboone_nested_bundle
        miniboone_stage05_split_10x1_X = miniboone_nested_X
        miniboone_stage05_split_10x1_y = miniboone_nested_y
        miniboone_stage05_split_10x1_features = miniboone_nested_features
    except NameError:
        miniboone_stage05_split_10x1_bundle = load_raw_candidate(
            STAGE05_SPLIT_10X1_CANDIDATE_ID
        )
        (
            miniboone_stage05_split_10x1_X,
            miniboone_stage05_split_10x1_features,
        ) = build_selected_numeric_frame(
            STAGE05_SPLIT_10X1_CANDIDATE_ID,
            miniboone_stage05_split_10x1_bundle["X_raw"],
        )
        miniboone_stage05_split_10x1_y = miniboone_stage05_split_10x1_bundle["y"]

stage05_split_10x1_execution_bundle = run_split_10x1_grid(
    contract=stage05_split_10x1_contract,
    candidate_bundle=miniboone_stage05_split_10x1_bundle,
    X=miniboone_stage05_split_10x1_X,
    y=miniboone_stage05_split_10x1_y,
    feature_names=miniboone_stage05_split_10x1_features,
    parameter_grid=stage05_split_10x1_parameter_grid,
    fixed_parameters=stage05_split_10x1_fixed_parameters,
    control_estimators=stage05_split_10x1_control_estimators,
)


In [ ]:
stage05_split_10x1_outer_scores_df = (
    stage05_split_10x1_execution_bundle.outer_scores
)
stage05_split_10x1_summary_df = stage05_split_10x1_execution_bundle.summary

write_csv_checked(
    stage05_split_10x1_outer_scores_df,
    MINIBOONE_STAGE05_SPLIT_10X1_OUTER_SCORES_PATH,
)
write_csv_checked(
    stage05_split_10x1_summary_df,
    MINIBOONE_STAGE05_SPLIT_10X1_SUMMARY_PATH,
)

print(MINIBOONE_STAGE05_SPLIT_10X1_OUTER_SCORES_PATH.relative_to(PROJECT_ROOT))
print(MINIBOONE_STAGE05_SPLIT_10X1_SUMMARY_PATH.relative_to(PROJECT_ROOT))
print("HGB parameter candidates:", len(list(ParameterGrid(stage05_split_10x1_parameter_grid))))
print("outer scores:", len(stage05_split_10x1_outer_scores_df))
print("summary rows:", len(stage05_split_10x1_summary_df))
display(
    stage05_split_10x1_summary_df[
        stage05_split_10x1_summary_df["comparison_model_id"].eq("logistic_regression")
        & stage05_split_10x1_summary_df["metric_name"].eq("average_precision")
    ]
)


## 11 — Этап 5: аудит конфликта метрик

Цель блока: выполнить заранее зафиксированный стресс-тест `ST05_04_metric_conflict_audit`.

Проверяется не новое обучение моделей, а согласованность уже полученных результатов по основной и вторичным метрикам.

Границы блока:

1. новые модели не обучаются;
2. используются только сохраненные сводки `ST05_02` и `ST05_03a`;
3. основная метрика — `average_precision`;
4. вторичные метрики — `roc_auc`, `f1`, `balanced_accuracy`, `log_loss`, `brier_score`;
5. для `log_loss` и `brier_score` учитывается направление «ниже лучше»;
6. итоговый вердикт этапа 5 этим блоком не выносится.

In [39]:
from mlcra.io import read_csv_checked, write_csv_checked
from mlcra.validation import require_columns
from mlcra.verdicts import build_metric_conflict_audit

MINIBOONE_STAGE05_METRIC_CONFLICT_ID = "ST05_04_metric_conflict_audit"
MINIBOONE_STAGE05_METRIC_CONFLICT_CLAIM_ID = "miniboone_hgb_vs_logreg_average_precision_v01"

MINIBOONE_STAGE05_METRIC_CONFLICT_PLAN_PATH = (
    PROJECT_ROOT
    / "configs"
    / "stress_tests"
    / "miniboone_stress_test_plan_v01.csv"
)
MINIBOONE_STAGE05_SEED_STABILITY_SUMMARY_PATH = (
    DATA_REGISTRY_DIR
    / "openml_miniboone_stage05_seed_stability_summary.csv"
)
MINIBOONE_STAGE05_SPLIT_10X1_SUMMARY_PATH = (
    DATA_REGISTRY_DIR
    / "openml_miniboone_stage05_split_10x1_summary.csv"
)
MINIBOONE_STAGE05_METRIC_CONFLICT_AUDIT_PATH = (
    DATA_REGISTRY_DIR
    / "openml_miniboone_stage05_metric_conflict_audit.csv"
)

required_stage05_metric_conflict_input_paths = [
    MINIBOONE_STAGE05_METRIC_CONFLICT_PLAN_PATH,
    MINIBOONE_STAGE05_SEED_STABILITY_SUMMARY_PATH,
    MINIBOONE_STAGE05_SPLIT_10X1_SUMMARY_PATH,
]

missing_stage05_metric_conflict_input_paths = [
    path for path in required_stage05_metric_conflict_input_paths
    if not path.exists()
]

if missing_stage05_metric_conflict_input_paths:
    raise FileNotFoundError(
        "Не найдены обязательные входные файлы ST05_04: "
        + "; ".join(str(path.relative_to(PROJECT_ROOT)) for path in missing_stage05_metric_conflict_input_paths)
    )

required_stage05_metric_conflict_plan_columns = [
    "stress_test_id",
    "requires_new_model_run",
    "family",
    "primary_metric",
    "secondary_metrics",
    "planned_outputs",
]

stage05_metric_conflict_plan = read_csv_checked(
    MINIBOONE_STAGE05_METRIC_CONFLICT_PLAN_PATH,
    required_columns=required_stage05_metric_conflict_plan_columns,
)
stage05_seed_stability_summary = read_csv_checked(MINIBOONE_STAGE05_SEED_STABILITY_SUMMARY_PATH)
stage05_split_10x1_summary = read_csv_checked(MINIBOONE_STAGE05_SPLIT_10X1_SUMMARY_PATH)

stage05_metric_conflict_plan_rows = stage05_metric_conflict_plan[
    stage05_metric_conflict_plan["stress_test_id"].eq(MINIBOONE_STAGE05_METRIC_CONFLICT_ID)
].copy()

if len(stage05_metric_conflict_plan_rows) != 1:
    raise ValueError(
        f"Ожидалась ровно одна строка stress_test_id={MINIBOONE_STAGE05_METRIC_CONFLICT_ID}, "
        f"получено: {len(stage05_metric_conflict_plan_rows)}"
    )

stage05_metric_conflict_plan_row = stage05_metric_conflict_plan_rows.iloc[0]

if stage05_metric_conflict_plan_row["requires_new_model_run"] != "no":
    raise ValueError("ST05_04 должен иметь requires_new_model_run=no")

if stage05_metric_conflict_plan_row["family"] != "metric_conflict":
    raise ValueError("ST05_04 должен иметь family=metric_conflict")

if stage05_metric_conflict_plan_row["primary_metric"] != "average_precision":
    raise ValueError("ST05_04 должен использовать primary_metric=average_precision")

expected_stage05_metric_conflict_secondary_metrics = [
    "roc_auc",
    "f1",
    "balanced_accuracy",
    "log_loss",
    "brier_score",
]
actual_stage05_metric_conflict_secondary_metrics = [
    value.strip()
    for value in str(stage05_metric_conflict_plan_row["secondary_metrics"]).split(";")
    if value.strip()
]

if actual_stage05_metric_conflict_secondary_metrics != expected_stage05_metric_conflict_secondary_metrics:
    raise ValueError(
        "secondary_metrics ST05_04 не совпадают с ожидаемым списком. "
        f"Ожидалось: {expected_stage05_metric_conflict_secondary_metrics}, "
        f"получено: {actual_stage05_metric_conflict_secondary_metrics}"
    )

expected_stage05_metric_conflict_outputs = {
    "openml_miniboone_stage05_metric_conflict_audit.csv",
}
actual_stage05_metric_conflict_outputs = {
    value.strip()
    for value in str(stage05_metric_conflict_plan_row["planned_outputs"]).split(";")
    if value.strip()
}

if actual_stage05_metric_conflict_outputs != expected_stage05_metric_conflict_outputs:
    raise ValueError(
        "planned_outputs ST05_04 не совпадает с ожидаемым набором. "
        f"Ожидалось: {sorted(expected_stage05_metric_conflict_outputs)}, "
        f"получено: {sorted(actual_stage05_metric_conflict_outputs)}"
    )

stage05_metric_conflict_audit_df = build_metric_conflict_audit(
    seed_summary_df=stage05_seed_stability_summary,
    split_summary_df=stage05_split_10x1_summary,
)

expected_stage05_metric_conflict_audit_rows = 19
if len(stage05_metric_conflict_audit_df) != expected_stage05_metric_conflict_audit_rows:
    raise ValueError(
        f"Неожиданное число строк ST05_04: "
        f"ожидалось {expected_stage05_metric_conflict_audit_rows}, "
        f"получено {len(stage05_metric_conflict_audit_df)}"
    )

write_csv_checked(
    stage05_metric_conflict_audit_df,
    MINIBOONE_STAGE05_METRIC_CONFLICT_AUDIT_PATH,
    index=False,
)

print("Сохранен файл ST05_04:")
print(MINIBOONE_STAGE05_METRIC_CONFLICT_AUDIT_PATH.relative_to(PROJECT_ROOT))

print("\nРазмер результата:")
print("Строки audit-файла:", len(stage05_metric_conflict_audit_df))

print("\nПроверка по всем протоколам и метрикам:")
display(
    stage05_metric_conflict_audit_df[
        stage05_metric_conflict_audit_df["audit_level"].isin(
            ["all_protocols_metric", "overall_metric_conflict_audit"]
        )
    ][
        [
            "audit_level",
            "metric_name",
            "metric_role",
            "n_blocks",
            "advantage_mean",
            "advantage_min",
            "advantage_max",
            "candidate_positive_blocks",
            "candidate_negative_blocks",
            "metric_conflict_status",
            "action_required",
        ]
    ].sort_values(
        ["audit_level", "metric_name"],
        kind="mergesort",
    )
)

print("\nКонтроль границ:")
print("1. Выполнен только заранее зафиксированный ST05_04.")
print("2. Новые модели не обучались.")
print("3. Использованы только сводки ST05_02 и ST05_03a.")
print("4. Проверена согласованность average_precision с вторичными метриками.")
print("5. dataset_candidate_registry.csv не изменялся.")
print("6. Итоговый вердикт этапа 5 этим блоком не выносится.")

Сохранен файл ST05_04:
data_registry\openml_miniboone_stage05_metric_conflict_audit.csv

Размер результата:
Строки audit-файла: 19

Проверка по всем протоколам и метрикам:


,audit_level,metric_name,metric_role,n_blocks,advantage_mean,advantage_min,advantage_max,candidate_positive_blocks,candidate_negative_blocks,metric_conflict_status,action_required
12,all_protocols_metric,average_precision,primary,80,0.091034,0.082056,0.10197,80,0,no_conflict,none
13,all_protocols_metric,balanced_accuracy,secondary,80,0.098349,0.093306,0.104759,80,0,no_conflict,none
14,all_protocols_metric,brier_score,secondary,80,0.04402,0.041322,0.047085,80,0,no_conflict,none
15,all_protocols_metric,f1,secondary,80,0.123347,0.115253,0.133108,80,0,no_conflict,none
16,all_protocols_metric,log_loss,secondary,80,0.144737,0.136084,0.155921,80,0,no_conflict,none
17,all_protocols_metric,roc_auc,secondary,80,0.045158,0.040531,0.050803,80,0,no_conflict,none
18,overall_metric_conflict_audit,all_primary_and_secondary_metrics,overall,80,,,,80,0,no_metric_conflict_detected,none



Контроль границ:
1. Выполнен только заранее зафиксированный ST05_04.
2. Новые модели не обучались.
3. Использованы только сводки ST05_02 и ST05_03a.
4. Проверена согласованность average_precision с вторичными метриками.
5. dataset_candidate_registry.csv не изменялся.
6. Итоговый вердикт этапа 5 этим блоком не выносится.


## 12 — Этап 5: устойчивость выбора гиперпараметров

Цель блока: выполнить заранее зафиксированный стресс-тест `ST05_05_parameter_selection_stability`.

Проверяется не новое качество модели и не новый протокол обучения, а устойчивость выбранных гиперпараметров `hist_gradient_boosting` по уже выполненным внешним блокам `ST05_02` и `ST05_03a`.

Границы блока:

1. новые модели не обучаются;
2. используются только сохраненные внешние оценки `ST05_02` и `ST05_03a`;
3. анализируются только строки `hist_gradient_boosting`;
4. источник параметров — `selected_params_json`;
5. проверяется частотность выбранных наборов параметров и значений отдельных параметров;
6. итоговый вердикт этапа 5 этим блоком не выносится.

In [48]:
from mlcra.io import read_csv_checked, write_csv_checked
from mlcra.verdicts import build_parameter_selection_stability_audit

MINIBOONE_STAGE05_PARAMETER_STABILITY_ID = "ST05_05_parameter_selection_stability"
MINIBOONE_STAGE05_PARAMETER_STABILITY_MODEL_ID = "hist_gradient_boosting"

MINIBOONE_STAGE05_PARAMETER_STABILITY_PLAN_PATH = (
    PROJECT_ROOT
    / "configs"
    / "stress_tests"
    / "miniboone_stress_test_plan_v01.csv"
)
MINIBOONE_STAGE05_SEED_STABILITY_OUTER_SCORES_PATH = (
    DATA_REGISTRY_DIR
    / "openml_miniboone_stage05_seed_stability_outer_scores.csv"
)
MINIBOONE_STAGE05_SPLIT_10X1_OUTER_SCORES_PATH = (
    DATA_REGISTRY_DIR
    / "openml_miniboone_stage05_split_10x1_outer_scores.csv"
)
MINIBOONE_STAGE05_PARAMETER_STABILITY_PATH = (
    DATA_REGISTRY_DIR
    / "openml_miniboone_stage05_parameter_selection_stability.csv"
)

required_stage05_parameter_stability_input_paths = [
    MINIBOONE_STAGE05_PARAMETER_STABILITY_PLAN_PATH,
    MINIBOONE_STAGE05_SEED_STABILITY_OUTER_SCORES_PATH,
    MINIBOONE_STAGE05_SPLIT_10X1_OUTER_SCORES_PATH,
]

missing_stage05_parameter_stability_input_paths = [
    path
    for path in required_stage05_parameter_stability_input_paths
    if not path.exists()
]

if missing_stage05_parameter_stability_input_paths:
    raise FileNotFoundError(
        "Не найдены обязательные входные файлы ST05_05: "
        + "; ".join(
            str(path.relative_to(PROJECT_ROOT))
            for path in missing_stage05_parameter_stability_input_paths
        )
    )

required_stage05_parameter_stability_plan_columns = [
    "stress_test_id",
    "requires_new_model_run",
    "family",
    "models",
    "primary_metric",
    "secondary_metrics",
    "planned_outputs",
]

stage05_parameter_stability_plan = read_csv_checked(
    MINIBOONE_STAGE05_PARAMETER_STABILITY_PLAN_PATH,
    required_columns=required_stage05_parameter_stability_plan_columns,
)
stage05_seed_stability_outer_scores = read_csv_checked(
    MINIBOONE_STAGE05_SEED_STABILITY_OUTER_SCORES_PATH,
)
stage05_split_10x1_outer_scores = read_csv_checked(
    MINIBOONE_STAGE05_SPLIT_10X1_OUTER_SCORES_PATH,
)

stage05_parameter_stability_plan_rows = stage05_parameter_stability_plan[
    stage05_parameter_stability_plan["stress_test_id"].eq(
        MINIBOONE_STAGE05_PARAMETER_STABILITY_ID
    )
].copy()

if len(stage05_parameter_stability_plan_rows) != 1:
    raise ValueError(
        "Ожидалась ровно одна строка "
        f"stress_test_id={MINIBOONE_STAGE05_PARAMETER_STABILITY_ID}, "
        f"получено: {len(stage05_parameter_stability_plan_rows)}"
    )

stage05_parameter_stability_plan_row = (
    stage05_parameter_stability_plan_rows.iloc[0]
)

if stage05_parameter_stability_plan_row["requires_new_model_run"] != "no":
    raise ValueError("ST05_05 должен иметь requires_new_model_run=no")

if stage05_parameter_stability_plan_row["family"] != "parameter_selection_stability":
    raise ValueError(
        "ST05_05 должен иметь family=parameter_selection_stability"
    )

if (
    stage05_parameter_stability_plan_row["models"]
    != MINIBOONE_STAGE05_PARAMETER_STABILITY_MODEL_ID
):
    raise ValueError(
        "ST05_05 должен анализировать только hist_gradient_boosting"
    )

if (
    stage05_parameter_stability_plan_row["primary_metric"]
    != "inner_best_average_precision"
):
    raise ValueError(
        "ST05_05 должен использовать "
        "primary_metric=inner_best_average_precision"
    )

if "selected_parameter_set_id frequency" not in str(
    stage05_parameter_stability_plan_row["secondary_metrics"]
):
    raise ValueError(
        "secondary_metrics ST05_05 должен включать "
        "selected_parameter_set_id frequency"
    )

expected_stage05_parameter_stability_outputs = {
    "openml_miniboone_stage05_parameter_selection_stability.csv",
}
actual_stage05_parameter_stability_outputs = {
    value.strip()
    for value in str(
        stage05_parameter_stability_plan_row["planned_outputs"]
    ).split(";")
    if value.strip()
}

if (
    actual_stage05_parameter_stability_outputs
    != expected_stage05_parameter_stability_outputs
):
    raise ValueError(
        "planned_outputs ST05_05 не совпадает с ожидаемым набором. "
        f"Ожидалось: "
        f"{sorted(expected_stage05_parameter_stability_outputs)}, "
        f"получено: "
        f"{sorted(actual_stage05_parameter_stability_outputs)}"
    )

stage05_parameter_stability_df = (
    build_parameter_selection_stability_audit(
        seed_outer_scores_df=stage05_seed_stability_outer_scores,
        split_outer_scores_df=stage05_split_10x1_outer_scores,
    )
)

expected_stage05_parameter_stability_rows = 22
if len(stage05_parameter_stability_df) != expected_stage05_parameter_stability_rows:
    raise ValueError(
        "Неожиданное число строк ST05_05: "
        f"ожидалось {expected_stage05_parameter_stability_rows}, "
        f"получено {len(stage05_parameter_stability_df)}"
    )

write_csv_checked(
    stage05_parameter_stability_df,
    MINIBOONE_STAGE05_PARAMETER_STABILITY_PATH,
    index=False,
)

print("Сохранен файл ST05_05:")
print(
    MINIBOONE_STAGE05_PARAMETER_STABILITY_PATH.relative_to(
        PROJECT_ROOT
    )
)

print("\nРазмер результата:")
print("Строки audit-файла:", len(stage05_parameter_stability_df))

print(
    "\nСводка устойчивости выбора параметров "
    "по всем завершенным протоколам:"
)
display(
    stage05_parameter_stability_df[
        stage05_parameter_stability_df["audit_level"].isin(
            [
                "all_protocols_parameter_set",
                "all_protocols_parameter_value",
                "overall_parameter_selection_stability",
            ]
        )
    ][
        [
            "audit_level",
            "parameter_name",
            "parameter_value",
            "selected_parameter_set_id",
            "n_blocks",
            "selection_count",
            "selection_share",
            "unique_parameter_set_count",
            "top_selection_share",
            "stability_status",
            "action_required",
        ]
    ].sort_values(
        [
            "audit_level",
            "parameter_name",
            "selection_count",
            "parameter_value",
        ],
        ascending=[True, True, False, True],
        kind="mergesort",
    )
)

print("\nКонтроль границ:")
print("1. Выполнен только заранее зафиксированный ST05_05.")
print("2. Новые модели не обучались.")
print("3. Использованы только внешние оценки ST05_02 и ST05_03a.")
print("4. Проанализированы только строки hist_gradient_boosting.")
print("5. dataset_candidate_registry.csv не изменялся.")
print("6. Итоговый вердикт этапа 5 этим блоком не выносится.")

Сохранен файл ST05_05:
data_registry\openml_miniboone_stage05_parameter_selection_stability.csv

Размер результата:
Строки audit-файла: 22

Сводка устойчивости выбора параметров по всем завершенным протоколам:


,audit_level,parameter_name,parameter_value,selected_parameter_set_id,n_blocks,selection_count,selection_share,unique_parameter_set_count,top_selection_share,stability_status,action_required
15,all_protocols_parameter_set,all_parameters,"{""l2_regularization"": 0.0, ""learning_rate"": 0....",hgb_b5168b128e78,80,40,0.5,2,0.5,,
14,all_protocols_parameter_set,all_parameters,"{""l2_regularization"": 0.01, ""learning_rate"": 0...",hgb_8316cb7622a5,80,40,0.5,2,0.5,,
16,all_protocols_parameter_value,l2_regularization,0.0,not_applicable,80,40,0.5,2,0.5,variable,document_instability
17,all_protocols_parameter_value,l2_regularization,0.01,not_applicable,80,40,0.5,2,0.5,variable,document_instability
18,all_protocols_parameter_value,learning_rate,0.1,not_applicable,80,80,1.0,1,1.0,stable,none
19,all_protocols_parameter_value,max_iter,150,not_applicable,80,80,1.0,1,1.0,stable,none
20,all_protocols_parameter_value,max_leaf_nodes,63,not_applicable,80,80,1.0,1,1.0,stable,none
21,overall_parameter_selection_stability,all_parameters,l2_regularization,overall,80,40,0.5,2,0.5,partially_stable,document_instability



Контроль границ:
1. Выполнен только заранее зафиксированный ST05_05.
2. Новые модели не обучались.
3. Использованы только внешние оценки ST05_02 и ST05_03a.
4. Проанализированы только строки hist_gradient_boosting.
5. dataset_candidate_registry.csv не изменялся.
6. Итоговый вердикт этапа 5 этим блоком не выносится.


## 13 — Этап 5: аудит вычислительной стоимости преимущества качества

Цель блока: выполнить заранее зафиксированный стресс-тест `ST05_06_cost_quality_audit`.

Проверяется не новое качество модели и не новый протокол обучения, а соотношение между приростом качества `hist_gradient_boosting` относительно `logistic_regression` и вычислительной стоимостью этого преимущества.

Границы блока:

1. новые модели не обучаются;
2. используются только сохраненные внешние оценки `ST05_02` и `ST05_03a`;
3. сравнение выполняется попарно внутри одинаковых внешних блоков;
4. анализируются поля стоимости `fit_seconds` и `predict_seconds`;
5. анализируются метрики `average_precision`, `roc_auc`, `f1`, `balanced_accuracy`, `log_loss`, `brier_score`;
6. итоговый вердикт этапа 5 этим блоком не выносится.

In [ ]:
from mlcra.io import read_csv_checked, write_csv_checked
from mlcra.verdicts import build_cost_quality_audit

MINIBOONE_STAGE05_COST_QUALITY_ID = "ST05_06_cost_quality_audit"
MINIBOONE_STAGE05_COST_QUALITY_CLAIM_ID = (
    "miniboone_hgb_vs_logreg_average_precision_v01"
)

MINIBOONE_STAGE05_COST_QUALITY_PLAN_PATH = (
    PROJECT_ROOT
    / "configs"
    / "stress_tests"
    / "miniboone_stress_test_plan_v01.csv"
)
MINIBOONE_STAGE05_COST_QUALITY_SEED_OUTER_SCORES_PATH = (
    DATA_REGISTRY_DIR
    / "openml_miniboone_stage05_seed_stability_outer_scores.csv"
)
MINIBOONE_STAGE05_COST_QUALITY_SPLIT_10X1_OUTER_SCORES_PATH = (
    DATA_REGISTRY_DIR
    / "openml_miniboone_stage05_split_10x1_outer_scores.csv"
)
MINIBOONE_STAGE05_COST_QUALITY_AUDIT_PATH = (
    DATA_REGISTRY_DIR
    / "openml_miniboone_stage05_cost_quality_audit.csv"
)

required_stage05_cost_quality_input_paths = [
    MINIBOONE_STAGE05_COST_QUALITY_PLAN_PATH,
    MINIBOONE_STAGE05_COST_QUALITY_SEED_OUTER_SCORES_PATH,
    MINIBOONE_STAGE05_COST_QUALITY_SPLIT_10X1_OUTER_SCORES_PATH,
]

missing_stage05_cost_quality_input_paths = [
    path
    for path in required_stage05_cost_quality_input_paths
    if not path.exists()
]

if missing_stage05_cost_quality_input_paths:
    raise FileNotFoundError(
        "Не найдены обязательные входные файлы ST05_06: "
        + "; ".join(
            str(path.relative_to(PROJECT_ROOT))
            for path in missing_stage05_cost_quality_input_paths
        )
    )

required_stage05_cost_quality_plan_columns = [
    "stress_test_id",
    "claim_id",
    "family",
    "requires_new_model_run",
    "run_group",
    "protocol_variant_id",
    "models",
    "primary_metric",
    "secondary_metrics",
    "parameter_space_policy",
    "comparison_unit",
    "planned_outputs",
]

stage05_cost_quality_plan = read_csv_checked(
    MINIBOONE_STAGE05_COST_QUALITY_PLAN_PATH,
    required_columns=required_stage05_cost_quality_plan_columns,
)
stage05_cost_quality_seed_outer_scores = read_csv_checked(
    MINIBOONE_STAGE05_COST_QUALITY_SEED_OUTER_SCORES_PATH,
)
stage05_cost_quality_split_10x1_outer_scores = read_csv_checked(
    MINIBOONE_STAGE05_COST_QUALITY_SPLIT_10X1_OUTER_SCORES_PATH,
)

stage05_cost_quality_plan_rows = stage05_cost_quality_plan[
    stage05_cost_quality_plan["stress_test_id"].eq(
        MINIBOONE_STAGE05_COST_QUALITY_ID
    )
].copy()

if len(stage05_cost_quality_plan_rows) != 1:
    raise ValueError(
        "Ожидалась ровно одна строка "
        f"stress_test_id={MINIBOONE_STAGE05_COST_QUALITY_ID}, "
        f"получено: {len(stage05_cost_quality_plan_rows)}"
    )

stage05_cost_quality_plan_row = stage05_cost_quality_plan_rows.iloc[0]

if (
    stage05_cost_quality_plan_row["claim_id"]
    != MINIBOONE_STAGE05_COST_QUALITY_CLAIM_ID
):
    raise ValueError(
        "ST05_06 должен относиться к claim_id="
        f"{MINIBOONE_STAGE05_COST_QUALITY_CLAIM_ID}"
    )

if stage05_cost_quality_plan_row["family"] != "cost_of_quality":
    raise ValueError("ST05_06 должен иметь family=cost_of_quality")

if stage05_cost_quality_plan_row["requires_new_model_run"] != "no":
    raise ValueError("ST05_06 должен иметь requires_new_model_run=no")

if stage05_cost_quality_plan_row["run_group"] != "derive_from_outer_scores":
    raise ValueError(
        "ST05_06 должен иметь run_group=derive_from_outer_scores"
    )

if (
    stage05_cost_quality_plan_row["protocol_variant_id"]
    != "all_completed_preregistered_variants"
):
    raise ValueError(
        "ST05_06 должен использовать "
        "protocol_variant_id=all_completed_preregistered_variants"
    )

expected_stage05_cost_quality_models = [
    "hist_gradient_boosting",
    "logistic_regression",
    "dummy_prior",
]
actual_stage05_cost_quality_models = [
    value.strip()
    for value in str(stage05_cost_quality_plan_row["models"]).split(";")
    if value.strip()
]

if actual_stage05_cost_quality_models != expected_stage05_cost_quality_models:
    raise ValueError(
        "models ST05_06 не совпадают с ожидаемым списком. "
        f"Ожидалось: {expected_stage05_cost_quality_models}, "
        f"получено: {actual_stage05_cost_quality_models}"
    )

if (
    stage05_cost_quality_plan_row["primary_metric"]
    != "average_precision per fit_seconds ratio"
):
    raise ValueError(
        "ST05_06 должен использовать primary_metric="
        "average_precision per fit_seconds ratio"
    )

expected_stage05_cost_quality_secondary_metrics = [
    "predict_seconds",
    "brier_score",
    "log_loss",
]
actual_stage05_cost_quality_secondary_metrics = [
    value.strip()
    for value in str(
        stage05_cost_quality_plan_row["secondary_metrics"]
    ).split(";")
    if value.strip()
]

if (
    actual_stage05_cost_quality_secondary_metrics
    != expected_stage05_cost_quality_secondary_metrics
):
    raise ValueError(
        "secondary_metrics ST05_06 не совпадают с ожидаемым списком. "
        f"Ожидалось: {expected_stage05_cost_quality_secondary_metrics}, "
        f"получено: {actual_stage05_cost_quality_secondary_metrics}"
    )

if (
    stage05_cost_quality_plan_row["parameter_space_policy"]
    != "no_model_refit;derive_only"
):
    raise ValueError(
        "ST05_06 должен использовать "
        "parameter_space_policy=no_model_refit;derive_only"
    )

if (
    stage05_cost_quality_plan_row["comparison_unit"]
    != "paired outer validation block"
):
    raise ValueError(
        "ST05_06 должен использовать "
        "comparison_unit=paired outer validation block"
    )

expected_stage05_cost_quality_outputs = {
    "openml_miniboone_stage05_cost_quality_audit.csv",
}
actual_stage05_cost_quality_outputs = {
    value.strip()
    for value in str(
        stage05_cost_quality_plan_row["planned_outputs"]
    ).split(";")
    if value.strip()
}

if (
    actual_stage05_cost_quality_outputs
    != expected_stage05_cost_quality_outputs
):
    raise ValueError(
        "planned_outputs ST05_06 не совпадает с ожидаемым набором. "
        f"Ожидалось: {sorted(expected_stage05_cost_quality_outputs)}, "
        f"получено: {sorted(actual_stage05_cost_quality_outputs)}"
    )

stage05_cost_quality_audit_df = build_cost_quality_audit(
    seed_outer_scores_df=stage05_cost_quality_seed_outer_scores,
    split_outer_scores_df=stage05_cost_quality_split_10x1_outer_scores,
)

expected_stage05_cost_quality_shape = (18, 40)
if stage05_cost_quality_audit_df.shape != expected_stage05_cost_quality_shape:
    raise ValueError(
        "Неожиданный размер результата ST05_06: "
        f"ожидалось {expected_stage05_cost_quality_shape}, "
        f"получено {stage05_cost_quality_audit_df.shape}"
    )

write_csv_checked(
    stage05_cost_quality_audit_df,
    MINIBOONE_STAGE05_COST_QUALITY_AUDIT_PATH,
    index=False,
)

print("Сохранен файл ST05_06:")
print(MINIBOONE_STAGE05_COST_QUALITY_AUDIT_PATH.relative_to(PROJECT_ROOT))

print("\nРазмер результата:")
print("Строки audit-файла:", len(stage05_cost_quality_audit_df))

print("\nСводка стоимости преимущества по average_precision:")
display(
    stage05_cost_quality_audit_df[
        stage05_cost_quality_audit_df["quality_metric"].eq(
            "average_precision"
        )
    ][
        [
            "protocol_scope_id",
            "paired_outer_blocks",
            "quality_delta_directional_mean",
            "quality_delta_directional_min",
            "quality_delta_directional_max",
            "candidate_fit_seconds_mean",
            "baseline_fit_seconds_mean",
            "fit_seconds_ratio_of_means",
            "candidate_predict_seconds_mean",
            "baseline_predict_seconds_mean",
            "predict_seconds_ratio_of_means",
            "candidate_total_seconds_mean",
            "baseline_total_seconds_mean",
            "total_seconds_ratio_of_means",
            "audit_status",
        ]
    ]
)

print("\nКонтроль границ:")
print("1. Выполнен только заранее зафиксированный ST05_06.")
print("2. Новые модели не обучались.")
print("3. Использованы только внешние оценки ST05_02 и ST05_03a.")
print(
    "4. Сравнение выполнено попарно: hist_gradient_boosting против "
    "logistic_regression внутри одинаковых внешних блоков."
)
print(
    "5. Проанализированы fit_seconds, predict_seconds и качество "
    "по шести метрикам."
)
print("6. dataset_candidate_registry.csv не изменялся.")
print("7. Итоговый вердикт этапа 5 этим блоком не выносится.")

Сохранен файл ST05_06:
data_registry\openml_miniboone_stage05_cost_quality_audit.csv

Размер результата:
Строки audit-файла: 18

Сводка стоимости преимущества по average_precision:


,protocol_scope_id,paired_outer_blocks,quality_delta_directional_mean,quality_delta_directional_min,quality_delta_directional_max,candidate_fit_seconds_mean,baseline_fit_seconds_mean,fit_seconds_ratio_of_means,candidate_predict_seconds_mean,baseline_predict_seconds_mean,predict_seconds_ratio_of_means,candidate_total_seconds_mean,baseline_total_seconds_mean,total_seconds_ratio_of_means,audit_status
0,ST05_02_seed_stability_grid,50,0.091739,0.086312,0.097726,247.571415,3.416245,72.468865,0.870690,0.072976,11.931258,248.442106,3.489221,71.202747,quality_advantage_with_high_training_cost_disc...
6,ST05_03a_split_protocol_10x1,30,0.090329,0.082056,0.101970,191.476628,5.250848,36.465850,0.702036,0.056232,12.484734,192.178664,5.307079,36.211756,quality_advantage_with_high_training_cost_disc...
12,all_completed_preregistered_variants,80,0.091210,0.082056,0.101970,226.535870,4.104221,55.195823,0.807445,0.066697,12.106246,227.343315,4.170918,54.506784,quality_advantage_with_high_training_cost_disc...



Контроль границ:
1. Выполнен только заранее зафиксированный ST05_06.
2. Новые модели не обучались.
3. Использованы только внешние оценки ST05_02 и ST05_03a.
4. Сравнение выполнено попарно: hist_gradient_boosting против logistic_regression внутри одинаковых внешних блоков.
5. Проанализированы fit_seconds, predict_seconds и качество по шести метрикам.
6. dataset_candidate_registry.csv не изменялся.
7. Итоговый вердикт этапа 5 этим блоком не выносится.


## 14 — Этап 5: проверка риска оптимистического смещения невложенной оценки

Цель блока: выполнить заранее зафиксированный стресс-тест `ST05_07_non_nested_optimism_probe`.

Проверяется не новое утверждение о качестве модели, а риск того, что невложенная процедура `GridSearchCV` может дать более оптимистическую оценку качества, чем вложенная внешняя проверка.

Границы блока:

1. выполняется новый запуск моделей только для `hist_gradient_boosting`;
2. используется только зафиксированная сетка параметров `same_locked_space_as_nested_cv_v01`;
3. основная метрика — `average_precision`;
4. невложенная оценка не используется как итоговая внешняя оценка качества;
5. результат сравнивается с уже сохраненными вложенными внешними оценками;
6. `dataset_candidate_registry.csv` не изменяется;
7. итоговый вердикт этапа 5 этим блоком не выносится.

In [ ]:
from mlcra.stress_tests import (
    build_non_nested_optimism_contract,
    build_non_nested_reference_table,
    run_non_nested_optimism_probe,
)

MINIBOONE_STAGE05_NON_NESTED_OPTIMISM_CLAIM_PATH = (
    PROJECT_ROOT / "configs" / "claims" / "miniboone_hgb_vs_logreg_claim_v01.csv"
)
MINIBOONE_STAGE05_NON_NESTED_OPTIMISM_PLAN_PATH = (
    PROJECT_ROOT / "configs" / "stress_tests" / "miniboone_stress_test_plan_v01.csv"
)
MINIBOONE_STAGE05_NON_NESTED_OPTIMISM_MODEL_SPACE_PATH = (
    PROJECT_ROOT / "configs" / "model_spaces"
    / "miniboone_hist_gradient_boosting_nested_space.csv"
)
MINIBOONE_STAGE05_NON_NESTED_OPTIMISM_NESTED_SUMMARY_PATH = (
    DATA_REGISTRY_DIR / "openml_miniboone_nested_summary.csv"
)
MINIBOONE_STAGE05_NON_NESTED_OPTIMISM_SEED_OUTER_SCORES_PATH = (
    DATA_REGISTRY_DIR / "openml_miniboone_stage05_seed_stability_outer_scores.csv"
)
MINIBOONE_STAGE05_NON_NESTED_OPTIMISM_SPLIT_10X1_OUTER_SCORES_PATH = (
    DATA_REGISTRY_DIR / "openml_miniboone_stage05_split_10x1_outer_scores.csv"
)
MINIBOONE_STAGE05_NON_NESTED_OPTIMISM_OUTPUT_PATH = (
    DATA_REGISTRY_DIR / "openml_miniboone_stage05_non_nested_optimism_probe.csv"
)

stage05_non_nested_plan = read_project_csv(MINIBOONE_STAGE05_NON_NESTED_OPTIMISM_PLAN_PATH)
stage05_non_nested_claim = read_project_csv(MINIBOONE_STAGE05_NON_NESTED_OPTIMISM_CLAIM_PATH)
stage05_non_nested_model_space = read_project_csv(MINIBOONE_STAGE05_NON_NESTED_OPTIMISM_MODEL_SPACE_PATH)
stage05_non_nested_nested_summary = read_project_csv(MINIBOONE_STAGE05_NON_NESTED_OPTIMISM_NESTED_SUMMARY_PATH)
stage05_non_nested_seed_outer_scores = read_project_csv(MINIBOONE_STAGE05_NON_NESTED_OPTIMISM_SEED_OUTER_SCORES_PATH)
stage05_non_nested_split_10x1_outer_scores = read_project_csv(MINIBOONE_STAGE05_NON_NESTED_OPTIMISM_SPLIT_10X1_OUTER_SCORES_PATH)

stage05_non_nested_contract = build_non_nested_optimism_contract(
    stage05_non_nested_plan, stage05_non_nested_claim
)
stage05_non_nested_reference_table = build_non_nested_reference_table(
    stage05_non_nested_nested_summary,
    stage05_non_nested_seed_outer_scores,
    stage05_non_nested_split_10x1_outer_scores,
    stage05_non_nested_contract,
)
stage05_non_nested_parameter_grid, stage05_non_nested_fixed_parameters = (
    build_hist_gradient_boosting_search_space(
        stage05_non_nested_model_space,
        protocol_id=stage05_non_nested_contract.nested_protocol_id,
        candidate_id=stage05_non_nested_contract.candidate_id,
    )
)

try:
    miniboone_stage05_non_nested_bundle = miniboone_stage05_split_10x1_bundle
    miniboone_stage05_non_nested_X = miniboone_stage05_split_10x1_X
    miniboone_stage05_non_nested_y = miniboone_stage05_split_10x1_y
    miniboone_stage05_non_nested_features = miniboone_stage05_split_10x1_features
except NameError:
    try:
        miniboone_stage05_non_nested_bundle = miniboone_stage05_seed_bundle
        miniboone_stage05_non_nested_X = miniboone_stage05_seed_X
        miniboone_stage05_non_nested_y = miniboone_stage05_seed_y
        miniboone_stage05_non_nested_features = miniboone_stage05_seed_features
    except NameError:
        try:
            miniboone_stage05_non_nested_bundle = miniboone_nested_bundle
            miniboone_stage05_non_nested_X = miniboone_nested_X
            miniboone_stage05_non_nested_y = miniboone_nested_y
            miniboone_stage05_non_nested_features = miniboone_nested_features
        except NameError:
            miniboone_stage05_non_nested_bundle = load_raw_candidate(
                stage05_non_nested_contract.candidate_id
            )
            (
                miniboone_stage05_non_nested_X,
                miniboone_stage05_non_nested_features,
            ) = build_selected_numeric_frame(
                stage05_non_nested_contract.candidate_id,
                miniboone_stage05_non_nested_bundle["X_raw"],
            )
            miniboone_stage05_non_nested_y = (
                miniboone_stage05_non_nested_bundle["y"]
            )

miniboone_stage05_non_nested_result = run_non_nested_optimism_probe(
    stage05_non_nested_contract,
    miniboone_stage05_non_nested_bundle,
    miniboone_stage05_non_nested_X,
    miniboone_stage05_non_nested_y,
    miniboone_stage05_non_nested_features,
    stage05_non_nested_parameter_grid,
    stage05_non_nested_fixed_parameters,
    stage05_non_nested_reference_table,
)
stage05_non_nested_optimism_df = miniboone_stage05_non_nested_result.audit
stage05_non_nested_optimism_df.to_csv(
    MINIBOONE_STAGE05_NON_NESTED_OPTIMISM_OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig",
)
print("Сохранен файл ST05_07:", MINIBOONE_STAGE05_NON_NESTED_OPTIMISM_OUTPUT_PATH.relative_to(PROJECT_ROOT))
print("Строки audit-файла:", len(stage05_non_nested_optimism_df))
display(stage05_non_nested_optimism_df)
print("ST05_07 остается diagnostic-only и не заменяет nested CV.")
